# 5.3.1 Validate, Interpret, and Package Counterfactual Results

5.2.1 measured how observed post-congestion-pricing mobility differed from the frozen no-CP forecast across time, place, mobility measure, and forecast horizon. This notebook asks the next question: **how robust are those main patterns when we make other reasonable analytical choices or compare the evidence on common support?**

5.3.1 tests a focused set of robustness questions, brings the relevant stability evidence into one review surface, freezes the reader-facing interpretation, and packages the final evidence for downstream use.

<img src="image-20260928-022353.png" width="" align="" />

## Notebook Setup

All notebook-wide imports, paths, frozen contracts, execution controls, and shared helpers are defined here before 5.3.1.1. The immediate source is the completed `5.2.1.final_tables` handoff: compact final summaries are reused directly, while the large row-level files are read only when a genuinely new robustness calculation requires them.

The first setup block loads the libraries and shared project branding used throughout 5.3.1. Display settings affect only notebook presentation; saved analytical values keep their full precision.

In [3]:
# ---------------------------------------------------------------------
# 5.3.1.0A — Imports, display defaults, and project branding
# ---------------------------------------------------------------------

from pathlib import Path

import gc
import os
import uuid
import shutil

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import plotly.graph_objects as go

from IPython.display import display

from project_branding import (
    BRAND_COLOR_SEQUENCE,
    BRAND_DIVERGING_SEQUENCE,
    BRAND_SEQUENTIAL_HEATMAP_SCALE,
    BRAND_COLORS,
    apply_branding,
)


# Keep notebook tables readable without changing the precision of values
# stored in memory or written to Parquet.
pd.options.display.float_format = "{:,.3f}".format
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

This block carries forward the analytical rules frozen in 5.2.1: five targets, three horizons, the no-CP-minus-observed gap direction, and the primary cross-regime population. These constants anchor the handoff checks and robustness analyses throughout 5.3.1.

In [2]:
# ---------------------------------------------------------------------
# 5.3.1.0B — Frozen analytical contracts and execution controls
# ---------------------------------------------------------------------

# Five mobility measures and three forecast horizons define the 15 jobs
# reviewed throughout this notebook.
FORECAST_TARGETS = [
    "taxi_trip_count", "taxi_avg_trip_speed",
    "fhvhv_trip_count", "fhvhv_avg_trip_speed",
    "subway_ridership",
]
FORECAST_HORIZONS = [1, 2, 5]

COUNT_TARGETS = ["taxi_trip_count", "fhvhv_trip_count", "subway_ridership"]
SPEED_TARGETS = ["taxi_avg_trip_speed", "fhvhv_avg_trip_speed"]

TARGET_LABELS = {
    "taxi_trip_count": "Taxi trips",
    "taxi_avg_trip_speed": "Taxi average speed",
    "fhvhv_trip_count": "FHVHV trips",
    "fhvhv_avg_trip_speed": "FHVHV average speed",
    "subway_ridership": "Subway ridership",
}
HORIZON_LABELS = {1: "h=1", 2: "h=2", 5: "h=5"}

# Frozen row counts and contract IDs anchor the 5.2.1 → 5.3.1 handoff.
EXPECTED_JOB_COUNT = 15
EXPECTED_GAP_ROWS = 7_023_640
EXPECTED_HORIZON_STABILITY_ROWS = 2_342_945
EXPECTED_TEMPORAL_SUMMARY_ROWS = 375
EXPECTED_SPATIAL_SUMMARY_ROWS = 3_309
EXPECTED_PROFILE_SUMMARY_ROWS = 3_459
EXPECTED_FEATURE_RELIANCE_ROWS = 86
EXPECTED_GAP_QA_ROWS = 15

GAP_CONTRACT_ID = "counterfactual_gap_estimation_v2"
HORIZON_STABILITY_CONTRACT_ID = "counterfactual_horizon_stability_v2"
FEATURE_RELIANCE_CONTRACT_ID = "counterfactual_h1_feature_reliance_v1"
UPSTREAM_HANDOFF_CONTRACT_ID = "counterfactual_gap_handoff_v2"
FINAL_HANDOFF_CONTRACT_ID = "counterfactual_validation_handoff_v1"

GAP_SIGN_CONVENTION = "no_cp_minus_observed"
PERCENT_GAP_DENOMINATOR = "observed_actual"

PRIMARY_POPULATION_LABEL = "Primary cross-regime"
NATIVE_POPULATION_LABEL = "Broader native forecasting"

# Intermediate artifacts provide restart points; the visual toggle controls
# chart rendering while leaving the analytical calculations unchanged.
WRITE_INTERMEDIATE_OUTPUTS = True
RUN_VISUALS = True

# ---------------------------------------------------------------------
# Portable Showcase packaging
# ---------------------------------------------------------------------
#
# This is separate from the analytical export controls because it only
# copies already-frozen final artifacts into one self-contained folder.
# Set False after the portable package has been created successfully.

WRITE_PORTABLE_SHOWCASE_PACKAGE = True

5.3.1 reads only from the completed `5.2.1.final_tables` package. Intermediate robustness summaries belong to this notebook, while the final validated handoff will be written later by 5.3.1's dedicated export chapter.

In [3]:
# ---------------------------------------------------------------------
# 5.3.1.0C — Authoritative input and working paths
# ---------------------------------------------------------------------

PROJECT_ROOT = Path.cwd()
PIPELINE_DIR = PROJECT_ROOT / "pipeline_data"

# 5.2.1 final tables are the authoritative counterfactual inputs.
INPUT_521_FINAL_DIR = PIPELINE_DIR / "5.2.1.final_tables"

GAP_SURFACE_PATH = INPUT_521_FINAL_DIR / "counterfactual_gap_surface.parquet"
HORIZON_STABILITY_SURFACE_PATH = (
    INPUT_521_FINAL_DIR / "counterfactual_horizon_stability.parquet"
)
TEMPORAL_SUMMARY_PATH = (
    INPUT_521_FINAL_DIR / "counterfactual_temporal_summary.parquet"
)
SPATIAL_SUMMARY_PATH = (
    INPUT_521_FINAL_DIR / "counterfactual_spatial_summary.parquet"
)
PROFILE_SUMMARY_PATH = (
    INPUT_521_FINAL_DIR / "counterfactual_mobility_profile_summary.parquet"
)
FEATURE_RELIANCE_PATH = (
    INPUT_521_FINAL_DIR / "counterfactual_h1_feature_reliance.parquet"
)

UPSTREAM_GAP_QA_PATH = INPUT_521_FINAL_DIR / "counterfactual_gap_qa.parquet"
UPSTREAM_HANDOFF_MANIFEST_PATH = (
    INPUT_521_FINAL_DIR / "counterfactual_gap_handoff_manifest.parquet"
)

# Restartable working artifacts stay separate from the final downstream handoff.
INTERMEDIATE_DIR = PIPELINE_DIR / "5.3.1.intermediate_tables"
FINAL_DIR = PIPELINE_DIR / "5.3.1.final_tables"

POPULATION_ROBUSTNESS_PATH = (
    INTERMEDIATE_DIR / "counterfactual_population_robustness.parquet"
)
SPEED_WEIGHTING_ROBUSTNESS_PATH = (
    INTERMEDIATE_DIR / "counterfactual_speed_weighting_robustness.parquet"
)
TEMPORAL_ROBUSTNESS_PATH = (
    INTERMEDIATE_DIR / "counterfactual_temporal_robustness.parquet"
)
GEOGRAPHY_ROBUSTNESS_PATH = (
    INTERMEDIATE_DIR / "counterfactual_geography_robustness.parquet"
)
HORIZON_ROBUSTNESS_PATH = (
    INTERMEDIATE_DIR / "counterfactual_horizon_robustness.parquet"
)
PRE_CP_CALIBRATION_PATH = (
    INTERMEDIATE_DIR / "counterfactual_pre_cp_error_calibration.parquet"
)
PREDECISION_EVIDENCE_PATH = (
    INTERMEDIATE_DIR / "counterfactual_predecision_evidence.parquet"
)
COUNTERFACTUAL_SHOWCASE_PORTABLE_MANIFEST_PATH = (
    FINAL_DIR
    / "counterfactual_showcase_portable_manifest.parquet"
)

FINAL_QA_PATH = FINAL_DIR / "counterfactual_validation_qa.parquet"
FINAL_MANIFEST_PATH = (
    FINAL_DIR / "counterfactual_validation_handoff_manifest.parquet"
)

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

These helpers keep repeated analytical rules consistent. They validate files and schemas, require the completed 5.2.1 handoff before robustness analysis begins, save and recover compact 5.3.1 restart artifacts safely, and preserve the 5.2.1 aggregation rules: counts add, speeds are trip-weighted, and percentage gaps are calculated only after aggregation.

In [4]:
# ---------------------------------------------------------------------
# 5.3.1.0D — Shared validation, persistence, and aggregation helpers
# ---------------------------------------------------------------------

def require_file(path, label):
    """Fail early when an authoritative input is missing."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"{label} is missing: {path}")

    return path


def require_columns(frame, required_columns, label):
    """
    Stop when a table no longer matches the fields this notebook expects.

    A renamed or missing field can otherwise produce a plausible-looking
    result built from the wrong information.
    """
    missing = sorted(set(required_columns) - set(frame.columns))

    if missing:
        raise KeyError(f"{label} is missing required columns: {missing}")


def parquet_metadata(path):
    """
    Inspect a Parquet file without loading its analytical rows.

    This is especially useful for the two multi-million-row 5.2.1 surfaces:
    row count and schema can be checked cheaply before targeted reads begin.
    """
    path = require_file(path, Path(path).name)
    parquet_file = pq.ParquetFile(path)
    column_names = list(parquet_file.schema_arrow.names)

    return {
        "rows": int(parquet_file.metadata.num_rows),
        "columns": len(column_names),
        "row_groups": int(parquet_file.metadata.num_row_groups),
        "size_mb": path.stat().st_size / (1024 ** 2),
        "columns_list": column_names,
    }


def qa_file_passes(path):
    """Return whether every check in an upstream QA table passes."""
    path = require_file(path, Path(path).name)
    frame = pd.read_parquet(path)

    if "passed" in frame.columns:
        return bool(frame["passed"].fillna(False).astype(bool).all()), frame

    if "status" in frame.columns:
        return bool(frame["status"].astype(str).eq("PASS").all()), frame

    raise KeyError(
        f"{path.name}: expected `passed` or `status` in the QA table."
    )


def atomic_write_parquet(frame, path):
    """
    Save through a temporary file before promoting the finished artifact.

    An interrupted write therefore cannot masquerade as a completed result
    when the notebook is restarted.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(
        f".{path.stem}.{uuid.uuid4().hex[:8]}.tmp.parquet"
    )

    try:
        frame.to_parquet(temporary_path, index=False, compression="zstd")
        os.replace(temporary_path, path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()


def load_saved_table(path, label, required_columns):
    """Recover a compact saved artifact and verify the fields we need."""
    frame = pd.read_parquet(require_file(path, label))
    require_columns(frame, required_columns, label)
    return frame


def require_upstream_handoff():
    """
    Require 5.3.1.1A to succeed before any robustness analysis runs.

    The large 5.2.1 surfaces are still read selectively later, but only after
    the complete final handoff has been validated as the authoritative input.
    """
    if not globals().get("UPSTREAM_HANDOFF_READY", False):
        raise RuntimeError(
            "Run 5.3.1.1A to load and validate the completed 5.2.1 "
            "counterfactual handoff before continuing."
        )


def aggregate_counterfactual_levels(
    frame,
    group_columns=None,
    speed_weight_mode="world_specific",
):
    """
    Aggregate completed 5.2.1 rows without changing their meaning.

    Counts and ridership add directly. Average speeds are weighted by trip
    activity. `common_observed` is used only for the speed sensitivity, where
    both worlds are evaluated over the observed trip mix.
    """
    group_columns = list(group_columns or [])
    result_parts = []

    for metric in FORECAST_TARGETS:
        metric_df = frame.loc[
            frame["metric"].eq(metric) & frame["gap_available"].astype(bool)
        ].copy()

        if metric_df.empty:
            continue

        group_keys = [*group_columns, "horizon"]

        if metric in COUNT_TARGETS:
            # Sum first, then calculate the percentage gap. This avoids
            # averaging row-level percentages across very different volumes.
            summary_df = (
                metric_df.groupby(group_keys, observed=True, dropna=False)
                .agg(
                    support_rows=("gap_available", "size"),
                    observed_level=("target_observed_value", "sum"),
                    no_cp_level=("system_prediction", "sum"),
                )
                .reset_index()
            )

        else:
            # Both speed definitions use the same rows so this sensitivity
            # changes only the weighting rule—not sample membership.
            valid_weights = (
                metric_df["observed_activity_weight"].notna()
                & metric_df["no_cp_activity_weight"].notna()
            )
            metric_df = metric_df.loc[valid_weights].copy()

            if speed_weight_mode == "world_specific":
                metric_df["observed_weight"] = metric_df["observed_activity_weight"]
                metric_df["no_cp_weight"] = metric_df["no_cp_activity_weight"]
            elif speed_weight_mode == "common_observed":
                metric_df["observed_weight"] = metric_df["observed_activity_weight"]
                metric_df["no_cp_weight"] = metric_df["observed_activity_weight"]
            else:
                raise ValueError(
                    "speed_weight_mode must be `world_specific` "
                    "or `common_observed`."
                )

            metric_df["observed_speed_x_trips"] = (
                metric_df["target_observed_value"] * metric_df["observed_weight"]
            )
            metric_df["no_cp_speed_x_trips"] = (
                metric_df["system_prediction"] * metric_df["no_cp_weight"]
            )

            summary_df = (
                metric_df.groupby(group_keys, observed=True, dropna=False)
                .agg(
                    support_rows=("gap_available", "size"),
                    observed_weighted_sum=("observed_speed_x_trips", "sum"),
                    observed_trip_sum=("observed_weight", "sum"),
                    no_cp_weighted_sum=("no_cp_speed_x_trips", "sum"),
                    no_cp_trip_sum=("no_cp_weight", "sum"),
                )
                .reset_index()
            )

            # Average speed is undefined when the corresponding activity sums
            # to zero; keep those cases missing rather than inventing 0 mph.
            summary_df["observed_level"] = (
                summary_df["observed_weighted_sum"]
                / summary_df["observed_trip_sum"].where(
                    summary_df["observed_trip_sum"] > 0
                )
            )
            summary_df["no_cp_level"] = (
                summary_df["no_cp_weighted_sum"]
                / summary_df["no_cp_trip_sum"].where(
                    summary_df["no_cp_trip_sum"] > 0
                )
            )
            summary_df = summary_df.drop(columns=[
                "observed_weighted_sum", "observed_trip_sum",
                "no_cp_weighted_sum", "no_cp_trip_sum",
            ])

        summary_df["metric"] = metric

        # Preserve the project-wide sign convention:
        # positive = no-CP mobility above observed mobility.
        summary_df["counterfactual_gap"] = (
            summary_df["no_cp_level"] - summary_df["observed_level"]
        )

        # Percentage gaps are calculated after aggregation. Genuine observed
        # zeros remain undefined instead of receiving an artificial percent.
        summary_df["counterfactual_gap_pct"] = np.where(
            summary_df["observed_level"].notna()
            & summary_df["observed_level"].ne(0),
            100 * summary_df["counterfactual_gap"] / summary_df["observed_level"],
            np.nan,
        )
        result_parts.append(summary_df)

    if not result_parts:
        return pd.DataFrame()

    ordered_columns = [
        *group_columns,
        "metric", "horizon", "support_rows",
        "observed_level", "no_cp_level",
        "counterfactual_gap", "counterfactual_gap_pct",
    ]
    sort_columns = [*group_columns, "metric", "horizon"]

    return (
        pd.concat(result_parts, ignore_index=True)[ordered_columns]
        .sort_values(sort_columns)
        .reset_index(drop=True)
    )


def read_gap_metric(metric, columns):
    """
    Read one mobility target and only the fields needed by the current test.

    This keeps each scan modest rather than loading the full seven-million-row
    counterfactual surface into memory.
    """
    require_upstream_handoff()
    requested_columns = list(dict.fromkeys(["metric", *columns]))

    return pd.read_parquet(
        GAP_SURFACE_PATH,
        columns=requested_columns,
        filters=[("metric", "==", metric)],
    )


def count_sign_flips(values):
    """
    Count genuine positive-to-negative or negative-to-positive reversals.

    Exact zeros are skipped because touching zero is not itself a reversal.
    """
    signs = np.sign(
        pd.to_numeric(values, errors="coerce").to_numpy(dtype=float)
    )
    signs = signs[np.isfinite(signs) & (signs != 0)]

    if len(signs) < 2:
        return 0

    return int(np.sum(signs[1:] != signs[:-1]))

## 5.3.1.1 Load and Validate the 5.2.1 Counterfactual Handoff

5.2.1 already built and validated the counterfactual gap evidence used here. Before testing robustness, confirm that its final handoff is intact and that all completed upstream QA checks still pass.

The compact temporal and spatial summaries are loaded because later sections use them directly. The much larger gap and horizon-stability surfaces remain on disk and are read in smaller pieces only when a robustness check needs them.

**5.3.1.1A — Load the completed 5.2.1 handoff.**

This step establishes the upstream boundary for the notebook. It confirms the frozen 5.2.1 final package and QA, loads the compact summaries reused below, and keeps the two large analytical surfaces available for targeted reads without loading millions of rows into memory at once.

In [5]:
# ---------------------------------------------------------------------
# 5.3.1.1A — Load and validate the completed 5.2.1 handoff
# ---------------------------------------------------------------------

# Readiness becomes true only after the expected handoff passes its boundary checks.
UPSTREAM_HANDOFF_READY = False


# ---------------------------------------------------------------------
# Confirm the expected 5.2.1 final artifacts
# ---------------------------------------------------------------------

required_521_files = {
    "Gap surface": GAP_SURFACE_PATH,
    "Horizon-stability surface": HORIZON_STABILITY_SURFACE_PATH,
    "Temporal summary": TEMPORAL_SUMMARY_PATH,
    "Spatial summary": SPATIAL_SUMMARY_PATH,
    "Multimodal profile summary": PROFILE_SUMMARY_PATH,
    "Feature reliance": FEATURE_RELIANCE_PATH,
    "Gap QA": UPSTREAM_GAP_QA_PATH,
    "Handoff manifest": UPSTREAM_HANDOFF_MANIFEST_PATH,
}

for label, path in required_521_files.items():
    require_file(path, f"5.2.1 {label}")


# ---------------------------------------------------------------------
# Confirm completed upstream QA
# ---------------------------------------------------------------------

upstream_qa_passed, upstream_gap_qa_df = qa_file_passes(
    UPSTREAM_GAP_QA_PATH
)

if len(upstream_gap_qa_df) != EXPECTED_GAP_QA_ROWS:
    raise RuntimeError(
        f"Expected {EXPECTED_GAP_QA_ROWS} completed 5.2.1 QA checks; "
        f"found {len(upstream_gap_qa_df)}."
    )

if not upstream_qa_passed:
    raise RuntimeError(
        "One or more completed 5.2.1 QA checks do not pass."
    )


# ---------------------------------------------------------------------
# Confirm the frozen handoff roster, paths, row counts, and contracts
# ---------------------------------------------------------------------

upstream_handoff_manifest_df = pd.read_parquet(
    UPSTREAM_HANDOFF_MANIFEST_PATH
)

if upstream_handoff_manifest_df["artifact_id"].duplicated().any():
    raise RuntimeError(
        "The 5.2.1 handoff manifest contains duplicate artifact IDs."
    )

expected_artifacts = {
    "counterfactual_gap_surface": {
        "path": GAP_SURFACE_PATH,
        "rows": EXPECTED_GAP_ROWS,
        "contract_id": GAP_CONTRACT_ID,
    },
    "counterfactual_horizon_stability": {
        "path": HORIZON_STABILITY_SURFACE_PATH,
        "rows": EXPECTED_HORIZON_STABILITY_ROWS,
        "contract_id": HORIZON_STABILITY_CONTRACT_ID,
    },
    "counterfactual_temporal_summary": {
        "path": TEMPORAL_SUMMARY_PATH,
        "rows": EXPECTED_TEMPORAL_SUMMARY_ROWS,
        "contract_id": GAP_CONTRACT_ID,
    },
    "counterfactual_spatial_summary": {
        "path": SPATIAL_SUMMARY_PATH,
        "rows": EXPECTED_SPATIAL_SUMMARY_ROWS,
        "contract_id": GAP_CONTRACT_ID,
    },
    "counterfactual_mobility_profile_summary": {
        "path": PROFILE_SUMMARY_PATH,
        "rows": EXPECTED_PROFILE_SUMMARY_ROWS,
        "contract_id": GAP_CONTRACT_ID,
    },
    "counterfactual_gap_qa": {
        "path": UPSTREAM_GAP_QA_PATH,
        "rows": EXPECTED_GAP_QA_ROWS,
        "contract_id": UPSTREAM_HANDOFF_CONTRACT_ID,
    },
    "counterfactual_h1_feature_reliance": {
        "path": FEATURE_RELIANCE_PATH,
        "rows": EXPECTED_FEATURE_RELIANCE_ROWS,
        "contract_id": FEATURE_RELIANCE_CONTRACT_ID,
    },
}

expected_artifact_ids = set(expected_artifacts)
observed_artifact_ids = set(
    upstream_handoff_manifest_df["artifact_id"].astype(str)
)

if observed_artifact_ids != expected_artifact_ids:
    missing = sorted(expected_artifact_ids - observed_artifact_ids)
    unexpected = sorted(observed_artifact_ids - expected_artifact_ids)

    raise RuntimeError(
        "The 5.2.1 handoff artifact roster has changed. "
        f"Missing={missing}; Unexpected={unexpected}"
    )

manifest_by_artifact = upstream_handoff_manifest_df.set_index("artifact_id")
inventory_rows = []

for artifact_id, expected in expected_artifacts.items():
    path = expected["path"]
    manifest_row = manifest_by_artifact.loc[artifact_id]
    actual_rows = int(pq.ParquetFile(path).metadata.num_rows)
    size_mb = path.stat().st_size / (1024 ** 2)
    manifest_path = Path(str(manifest_row["path"]))

    if manifest_path.resolve() != path.resolve():
        raise RuntimeError(
            f"{artifact_id}: manifest path does not match the frozen "
            "5.2.1 final-table path."
        )

    if int(manifest_row["rows"]) != expected["rows"]:
        raise RuntimeError(
            f"{artifact_id}: manifest row count changed. "
            f"Expected {expected['rows']:,}; "
            f"found {int(manifest_row['rows']):,}."
        )

    if actual_rows != expected["rows"]:
        raise RuntimeError(
            f"{artifact_id}: Parquet row count changed. "
            f"Expected {expected['rows']:,}; found {actual_rows:,}."
        )

    if str(manifest_row["contract_id"]) != expected["contract_id"]:
        raise RuntimeError(
            f"{artifact_id}: expected contract "
            f"`{expected['contract_id']}` but found "
            f"`{manifest_row['contract_id']}`."
        )

    inventory_rows.append({
        "Artifact": artifact_id,
        "Rows": actual_rows,
        "Size MB": size_mb,
        "Contract": expected["contract_id"],
    })


# ---------------------------------------------------------------------
# Load the compact summaries reused throughout 5.3.1
# ---------------------------------------------------------------------

temporal_summary_df = pd.read_parquet(TEMPORAL_SUMMARY_PATH)
spatial_summary_df = pd.read_parquet(SPATIAL_SUMMARY_PATH)

UPSTREAM_HANDOFF_READY = True


# ---------------------------------------------------------------------
# Reader-facing confirmation
# ---------------------------------------------------------------------

handoff_inventory_df = pd.DataFrame(inventory_rows)

loaded_inputs_df = pd.DataFrame([
    {
        "Input": "Temporal summary",
        "Rows loaded": len(temporal_summary_df),
        "Used by": "Temporal consistency",
    },
    {
        "Input": "Spatial summary",
        "Rows loaded": len(spatial_summary_df),
        "Used by": "Geographic consistency",
    },
    {
        "Input": "Gap surface",
        "Rows loaded": 0,
        "Used by": "Targeted sensitivity and calibration reads",
    },
    {
        "Input": "Horizon-stability surface",
        "Rows loaded": 0,
        "Used by": "Cross-horizon robustness",
    },
])

display(handoff_inventory_df.round({"Size MB": 3}))
display(loaded_inputs_df)

,Artifact,Rows,Size MB,Contract
0,counterfactual_gap_surface,7023640,286.249,counterfactual_gap_estimation_v2
1,counterfactual_horizon_stability,2342945,184.563,counterfactual_horizon_stability_v2
2,counterfactual_temporal_summary,375,0.018,counterfactual_gap_estimation_v2
3,counterfactual_spatial_summary,3309,0.114,counterfactual_gap_estimation_v2
4,counterfactual_mobility_profile_summary,3459,0.043,counterfactual_gap_estimation_v2
5,counterfactual_gap_qa,15,0.003,counterfactual_gap_handoff_v2
6,counterfactual_h1_feature_reliance,86,0.007,counterfactual_h1_feature_reliance_v1


,Input,Rows loaded,Used by
0,Temporal summary,375,Temporal consistency
1,Spatial summary,3309,Geographic consistency
2,Gap surface,0,Targeted sensitivity and calibration reads
3,Horizon-stability surface,0,Cross-horizon robustness


**Findings.** The complete 5.2.1 handoff is intact: all **seven authoritative artifacts** match their expected final paths, row counts, and contract IDs, and all **15 upstream QA checks pass**.

The **375-row temporal** and **3,309-row spatial** summaries are loaded for reuse, while the **7,023,640-row gap surface** and **2,342,945-row horizon-stability surface** remain on disk for targeted reads. The **86-row h=1 feature-reliance comparison** is retained as part of the authoritative 5.2.1 package but is not needed for the robustness calculations in this notebook.

## 5.3.1.2 Test Population and Forecast-Support Robustness

The primary cross-regime population remains the main counterfactual population because its support rule was frozen before the post-CP gaps were observed. The broader native forecasting population is used only as a sensitivity check: **would including those additional forecastable series materially change the headline result?**

**5.3.1.2A — Compare the primary result with the broader native population.**

For every Target × Horizon, calculate the full post-CP gap once on the frozen primary population and once across all native rows where an observed-versus-no-CP gap is available. Both versions use the same 5.2.1 aggregation rules, so any difference comes from population choice rather than a different gap formula.

In [6]:
# ---------------------------------------------------------------------
# 5.3.1.2A — Compare primary and broader native populations
# ---------------------------------------------------------------------

require_upstream_handoff()

population_columns = [
    "horizon", "system_prediction", "target_observed_value",
    "gap_available", "primary_gap_eligible",
    "observed_activity_weight", "no_cp_activity_weight",
]
population_parts = []


# Read one mobility target at a time so this sensitivity never needs the
# full seven-million-row gap surface in memory.
for metric in FORECAST_TARGETS:
    metric_df = read_gap_metric(metric, population_columns)
    require_columns(
        metric_df,
        ["metric", *population_columns],
        f"{TARGET_LABELS[metric]} gap rows",
    )
    metric_df["horizon"] = metric_df["horizon"].astype(int)

    # Primary is the frozen 5.2.1 interpretation. Native adds every row where
    # both the no-CP forecast and observed outcome support a valid gap.
    population_masks = {
        PRIMARY_POPULATION_LABEL:
            metric_df["primary_gap_eligible"].fillna(False).astype(bool),
        NATIVE_POPULATION_LABEL:
            metric_df["gap_available"].fillna(False).astype(bool),
    }

    for population_label, population_mask in population_masks.items():
        population_df = metric_df.loc[population_mask].copy()

        if population_df.empty:
            raise RuntimeError(
                f"{TARGET_LABELS[metric]}: `{population_label}` "
                "contains no usable rows."
            )

        # Use the same 5.2.1 aggregation rules in both populations so
        # population membership is the only analytical choice that changes.
        summary_df = aggregate_counterfactual_levels(
            population_df,
            speed_weight_mode="world_specific",
        )

        if summary_df.empty:
            raise RuntimeError(
                f"{TARGET_LABELS[metric]}: aggregation returned no rows "
                f"for `{population_label}`."
            )

        summary_df["population"] = population_label
        population_parts.append(summary_df)

    del metric_df, population_df
    gc.collect()


# Five targets × three horizons × two population definitions = 30 summaries.
population_summary_df = pd.concat(population_parts, ignore_index=True)

if len(population_summary_df) != EXPECTED_JOB_COUNT * 2:
    raise RuntimeError(
        "Population robustness did not produce the expected 30 "
        "Target × Horizon × Population rows."
    )

if population_summary_df[
    ["metric", "horizon", "population"]
].duplicated().any():
    raise RuntimeError(
        "Population robustness contains duplicate "
        "Target × Horizon × Population rows."
    )


# Put the two population versions on the same Target × Horizon row so both
# direction reversals and magnitude shifts are immediately visible.
primary_population_df = (
    population_summary_df.loc[
        population_summary_df["population"].eq(PRIMARY_POPULATION_LABEL)
    ]
    .drop(columns="population")
    .rename(columns={
        "support_rows": "primary_support_rows",
        "observed_level": "primary_observed_level",
        "no_cp_level": "primary_no_cp_level",
        "counterfactual_gap": "primary_gap",
        "counterfactual_gap_pct": "primary_gap_pct",
    })
)

native_population_df = (
    population_summary_df.loc[
        population_summary_df["population"].eq(NATIVE_POPULATION_LABEL)
    ]
    .drop(columns="population")
    .rename(columns={
        "support_rows": "native_support_rows",
        "observed_level": "native_observed_level",
        "no_cp_level": "native_no_cp_level",
        "counterfactual_gap": "native_gap",
        "counterfactual_gap_pct": "native_gap_pct",
    })
)

population_robustness_df = primary_population_df.merge(
    native_population_df,
    on=["metric", "horizon"],
    how="inner",
    validate="one_to_one",
)

# Direction and magnitude answer different robustness questions, so preserve
# them separately rather than collapsing them into one score.
population_robustness_df["same_direction"] = (
    np.sign(population_robustness_df["primary_gap"])
    == np.sign(population_robustness_df["native_gap"])
)
population_robustness_df["gap_pct_shift_pp"] = (
    population_robustness_df["native_gap_pct"]
    - population_robustness_df["primary_gap_pct"]
)
population_robustness_df["absolute_gap_pct_shift_pp"] = (
    population_robustness_df["gap_pct_shift_pp"].abs()
)
population_robustness_df["Target"] = (
    population_robustness_df["metric"].map(TARGET_LABELS)
)

population_robustness_df = (
    population_robustness_df
    .sort_values(["metric", "horizon"])
    .reset_index(drop=True)
)

if len(population_robustness_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Population robustness does not contain all 15 Target × Horizon jobs."
    )

if population_robustness_df[["metric", "horizon"]].duplicated().any():
    raise RuntimeError(
        "Population robustness contains duplicate Target × Horizon jobs."
    )

if WRITE_INTERMEDIATE_OUTPUTS:
    atomic_write_parquet(
        population_robustness_df,
        POPULATION_ROBUSTNESS_PATH,
    )


# Compact table: enough information to judge the sensitivity without
# repeating the much larger row-level counterfactual surface.
population_display_df = (
    population_robustness_df[
        [
            "Target", "horizon",
            "primary_support_rows", "native_support_rows",
            "primary_gap_pct", "native_gap_pct",
            "gap_pct_shift_pp", "same_direction",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "primary_support_rows": "Primary rows",
        "native_support_rows": "Native rows",
        "primary_gap_pct": "Primary gap %",
        "native_gap_pct": "Native gap %",
        "gap_pct_shift_pp": "Shift (pp)",
        "same_direction": "Same direction",
    })
)

display(population_display_df.round(3))


# ---------------------------------------------------------------------
# Visual check — where does population choice move the result most?
# ---------------------------------------------------------------------

population_heatmap_df = (
    population_robustness_df.pivot(
        index="metric",
        columns="horizon",
        values="gap_pct_shift_pp",
    )
    .reindex(index=FORECAST_TARGETS, columns=FORECAST_HORIZONS)
)
population_heatmap_values = population_heatmap_df.to_numpy(dtype=float)
population_color_limit = np.nanmax(np.abs(population_heatmap_values))

if not np.isfinite(population_color_limit) or population_color_limit == 0:
    population_color_limit = 1.0

population_fig = go.Figure(
    go.Heatmap(
        z=population_heatmap_values,
        x=[HORIZON_LABELS[h] for h in FORECAST_HORIZONS],
        y=[TARGET_LABELS[m] for m in FORECAST_TARGETS],
        colorscale=BRAND_DIVERGING_SEQUENCE,
        zmin=-population_color_limit,
        zmax=population_color_limit,
        zmid=0,
        colorbar={"title": "Shift<br>(pp)", "tickformat": ".3f"},
        text=np.round(population_heatmap_values, 3),
        texttemplate="%{text:+.3f}",
        hovertemplate=(
            "<b>%{y}</b><br>%{x}<br>"
            "Native − primary gap: %{z:+.3f} pp<extra></extra>"
        ),
    )
)

apply_branding(population_fig)
population_fig.update_layout(
    title={
        "text": (
            "<b>Does the Broader Native Population Change the Result?</b>"
            "<br><sup>Native percentage gap minus primary cross-regime "
            "percentage gap</sup>"
        ),
        "x": 0.5,
    },
    width=820,
    height=470,
    margin={"l": 150, "r": 70, "t": 100, "b": 55},
    xaxis={"title": "Forecast horizon"},
    yaxis={"title": ""},
)

if RUN_VISUALS:
    population_fig.show()

,Target,Horizon,Primary rows,Native rows,Primary gap %,Native gap %,Shift (pp),Same direction
0,FHVHV average speed,1,576386,578291,6.200,6.202,0.001,True
1,FHVHV average speed,2,576113,578013,6.257,6.258,0.001,True
2,FHVHV average speed,5,575350,577248,6.866,6.868,0.001,True
3,FHVHV trips,1,578785,580710,-10.997,-10.993,0.004,True
4,FHVHV trips,2,578512,580432,-11.028,-11.024,0.004,True
5,FHVHV trips,5,577745,579663,-11.028,-11.024,0.004,True
6,Subway ridership,1,334434,341138,-14.231,-14.261,-0.030,True
7,Subway ridership,2,334280,340980,-14.417,-14.447,-0.030,True
8,Subway ridership,5,333835,340523,-14.443,-14.473,-0.030,True
9,Taxi average speed,1,247181,247181,1.308,1.308,0.000,True


**Findings.** Using the broader set of forecastable Taxi Zones produces essentially the same result as the main analysis. All 15 Target × Horizon results keep the same direction, and the estimated gaps barely change—the largest difference is only **0.030 percentage points**, for Subway ridership. In practical terms, our conclusions are not being driven by which of the supported Taxi Zones we include.

## 5.3.1.3 Test Average-Speed Weighting Robustness

Average speed depends partly on which trips receive more weight. The 5.2.1 baseline lets the observed and no-CP worlds use their own trip activity; this sensitivity holds the trip mix fixed by using observed activity for both worlds and asks whether the Taxi and FHVHV speed conclusions materially change.

**5.3.1.3A — Compare world-specific and common observed-trip weighting.**

Both calculations use the exact same primary-population rows. Only the no-CP activity weights change, allowing us to separate the speed difference itself from changes in which trips make up the observed and no-CP worlds.

In [7]:
# ---------------------------------------------------------------------
# 5.3.1.3A — Compare world-specific and common-observed speed weights
# ---------------------------------------------------------------------

require_upstream_handoff()

speed_columns = [
    "horizon", "system_prediction", "target_observed_value",
    "gap_available", "primary_gap_eligible",
    "observed_activity_weight", "no_cp_activity_weight",
]
speed_weighting_parts = []


for metric in SPEED_TARGETS:
    metric_df = read_gap_metric(metric, speed_columns)
    require_columns(
        metric_df,
        ["metric", *speed_columns],
        f"{TARGET_LABELS[metric]} speed-gap rows",
    )
    metric_df["horizon"] = metric_df["horizon"].astype(int)

    # Both weighting rules must use exactly the same rows. Otherwise a
    # difference could come from sample membership instead of trip weighting.
    common_support_mask = (
        metric_df["primary_gap_eligible"].fillna(False).astype(bool)
        & metric_df["observed_activity_weight"].notna()
        & metric_df["no_cp_activity_weight"].notna()
    )
    common_support_df = metric_df.loc[common_support_mask].copy()

    if common_support_df.empty:
        raise RuntimeError(
            f"{TARGET_LABELS[metric]} has no common-support speed rows."
        )

    # Baseline: each world is weighted by its own trip activity.
    world_specific_df = aggregate_counterfactual_levels(
        common_support_df,
        speed_weight_mode="world_specific",
    )
    world_specific_df["weighting"] = "World-specific"

    # Sensitivity: both worlds are evaluated over the observed trip mix.
    common_observed_df = aggregate_counterfactual_levels(
        common_support_df,
        speed_weight_mode="common_observed",
    )
    common_observed_df["weighting"] = "Common observed"

    speed_weighting_parts.extend([world_specific_df, common_observed_df])

    del metric_df, common_support_df
    gc.collect()


speed_weighting_summary_df = pd.concat(
    speed_weighting_parts,
    ignore_index=True,
)

expected_speed_rows = (
    len(SPEED_TARGETS)
    * len(FORECAST_HORIZONS)
    * 2
)

if len(speed_weighting_summary_df) != expected_speed_rows:
    raise RuntimeError(
        "Speed-weighting robustness did not produce the expected "
        f"{expected_speed_rows} Target × Horizon × Weighting rows."
    )

world_specific_df = (
    speed_weighting_summary_df.loc[
        speed_weighting_summary_df["weighting"].eq("World-specific")
    ]
    .drop(columns="weighting")
    .rename(columns={
        "support_rows": "common_support_rows",
        "observed_level": "world_specific_observed_level",
        "no_cp_level": "world_specific_no_cp_level",
        "counterfactual_gap": "world_specific_gap",
        "counterfactual_gap_pct": "world_specific_gap_pct",
    })
)

common_observed_df = (
    speed_weighting_summary_df.loc[
        speed_weighting_summary_df["weighting"].eq("Common observed")
    ]
    .drop(columns=["weighting", "support_rows"])
    .rename(columns={
        "observed_level": "common_observed_observed_level",
        "no_cp_level": "common_observed_no_cp_level",
        "counterfactual_gap": "common_observed_gap",
        "counterfactual_gap_pct": "common_observed_gap_pct",
    })
)

speed_weighting_robustness_df = world_specific_df.merge(
    common_observed_df,
    on=["metric", "horizon"],
    how="inner",
    validate="one_to_one",
)

# A sign reversal is qualitatively different from a shift in magnitude, so
# preserve both pieces of evidence.
speed_weighting_robustness_df["same_direction"] = (
    np.sign(speed_weighting_robustness_df["world_specific_gap"])
    == np.sign(speed_weighting_robustness_df["common_observed_gap"])
)
speed_weighting_robustness_df["gap_pct_shift_pp"] = (
    speed_weighting_robustness_df["common_observed_gap_pct"]
    - speed_weighting_robustness_df["world_specific_gap_pct"]
)
speed_weighting_robustness_df["Target"] = (
    speed_weighting_robustness_df["metric"].map(TARGET_LABELS)
)

if WRITE_INTERMEDIATE_OUTPUTS:
    atomic_write_parquet(
        speed_weighting_robustness_df,
        SPEED_WEIGHTING_ROBUSTNESS_PATH,
    )

speed_weighting_display_df = (
    speed_weighting_robustness_df[
        [
            "Target", "horizon", "common_support_rows",
            "world_specific_gap_pct", "common_observed_gap_pct",
            "gap_pct_shift_pp", "same_direction",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "common_support_rows": "Common rows",
        "world_specific_gap_pct": "World-specific gap %",
        "common_observed_gap_pct": "Common-weight gap %",
        "gap_pct_shift_pp": "Shift (pp)",
        "same_direction": "Same direction",
    })
)

display(speed_weighting_display_df.round(3))


# ---------------------------------------------------------------------
# Visual check — does the speed story move when trip mix is fixed?
# ---------------------------------------------------------------------

speed_plot_df = speed_weighting_robustness_df.copy()
speed_plot_df["Job"] = (
    speed_plot_df["Target"]
    + " · "
    + speed_plot_df["horizon"].map(HORIZON_LABELS)
)

speed_fig = go.Figure()
speed_fig.add_trace(
    go.Bar(
        x=speed_plot_df["Job"],
        y=speed_plot_df["world_specific_gap_pct"],
        name="World-specific weights",
        marker={"color": BRAND_COLOR_SEQUENCE[0]},
        hovertemplate="<b>%{x}</b><br>Gap: %{y:+.3f}%<extra></extra>",
    )
)
speed_fig.add_trace(
    go.Bar(
        x=speed_plot_df["Job"],
        y=speed_plot_df["common_observed_gap_pct"],
        name="Common observed weights",
        marker={"color": BRAND_COLOR_SEQUENCE[2]},
        hovertemplate="<b>%{x}</b><br>Gap: %{y:+.3f}%<extra></extra>",
    )
)

apply_branding(speed_fig)
speed_fig.update_layout(
    title={
        "text": (
            "<b>Do Average-Speed Conclusions Depend on the Activity Weights?</b>"
            "<br><sup>Primary population · exact common rows</sup>"
        ),
        "x": 0.5,
    },
    barmode="group",
    width=980,
    height=520,
    margin={"l": 70, "r": 35, "t": 105, "b": 105},
    xaxis={"title": "Target × Horizon"},
    yaxis={"title": "No-CP − observed (%)", "zeroline": True},
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.01,
        "xanchor": "right",
        "x": 1.0,
    },
)

if RUN_VISUALS:
    speed_fig.show()

,Target,Horizon,Common rows,World-specific gap %,Common-weight gap %,Shift (pp),Same direction
0,Taxi average speed,1,247181,1.308,3.181,1.873,True
1,Taxi average speed,2,247078,12.193,14.256,2.063,True
2,Taxi average speed,5,246756,5.399,7.336,1.937,True
3,FHVHV average speed,1,576386,6.200,5.836,-0.365,True
4,FHVHV average speed,2,576113,6.257,6.005,-0.252,True
5,FHVHV average speed,5,575350,6.866,6.481,-0.385,True


**Findings.** Both speed measures keep the same basic story when Taxi Zones are weighted using the same observed trip activity on both sides of the comparison. FHVHV speed barely changes, while the Taxi-speed gaps increase by about **1.9 to 2.1 percentage points**. That does not reverse the Taxi-speed result, but it does show that the exact size of the Taxi-speed difference depends somewhat on how trip activity is weighted.

## 5.3.1.4 Test Stability Across Time, Geography, and Forecast Horizon

This section turns the completed monthly and horizon evidence into compact robustness measures and adds a broader geographic-consistency check.

Together, these views show whether each headline result remains broadly consistent through time, across larger parts of the city, and across the three forecast horizons.

**5.3.1.4A — Summarize the monthly consistency established in 5.2.1.**

Use the exported monthly gaps to reduce the temporal evidence to a few comparable measures for each Target × Horizon: how often the monthly direction matches the full-period result, the range of monthly percentage gaps, and the number of genuine direction reversals.

In [8]:
# ---------------------------------------------------------------------
# 5.3.1.4A — Summarize temporal consistency from the 5.2.1 final table
# ---------------------------------------------------------------------

require_upstream_handoff()

# Use the saved system-wide primary results when the in-memory copy is absent.
if "population_robustness_df" not in globals():
    population_robustness_df = load_saved_table(
        POPULATION_ROBUSTNESS_PATH,
        "5.3.1 population robustness",
        ["metric", "horizon", "primary_gap"],
    )

# The month-level summary is the authoritative input for temporal persistence.
monthly_summary_df = temporal_summary_df.loc[
    temporal_summary_df["summary_grain"].eq("month")
].copy()
monthly_summary_df["target_month"] = pd.to_datetime(
    monthly_summary_df["target_month"]
)

if monthly_summary_df.empty:
    raise RuntimeError(
        "The validated 5.2.1 temporal handoff contains no monthly rows."
    )

temporal_robustness_rows = []

for overall_row in population_robustness_df.itertuples(index=False):
    metric = overall_row.metric
    horizon = int(overall_row.horizon)

    month_df = (
        monthly_summary_df.loc[
            monthly_summary_df["metric"].eq(metric)
            & monthly_summary_df["horizon"].eq(horizon)
        ]
        .sort_values("target_month")
        .copy()
    )

    if month_df.empty:
        raise RuntimeError(
            f"{TARGET_LABELS[metric]} h={horizon}: no monthly rows were found."
        )

    overall_sign = np.sign(overall_row.primary_gap)
    month_signs = np.sign(
        month_df["counterfactual_gap"].to_numpy(dtype=float)
    )

    # Agreement measures how often monthly direction matches the full-period
    # result; sign flips count genuine reversals through time.
    monthly_agreement_pct = (
        100 * np.mean(month_signs == overall_sign)
        if overall_sign != 0
        else np.nan
    )

    temporal_robustness_rows.append({
        "metric": metric,
        "horizon": horizon,
        "month_count": len(month_df),
        "monthly_direction_agreement_pct": monthly_agreement_pct,
        "monthly_sign_flips": count_sign_flips(
            month_df["counterfactual_gap"]
        ),
        "monthly_gap_pct_min": month_df["counterfactual_gap_pct"].min(),
        "monthly_gap_pct_max": month_df["counterfactual_gap_pct"].max(),
    })

temporal_robustness_df = (
    pd.DataFrame(temporal_robustness_rows)
    .sort_values(["metric", "horizon"])
    .reset_index(drop=True)
)

if len(temporal_robustness_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Temporal robustness did not produce all 15 Target × Horizon jobs."
    )

if temporal_robustness_df[["metric", "horizon"]].duplicated().any():
    raise RuntimeError(
        "Temporal robustness contains duplicate Target × Horizon jobs."
    )

temporal_robustness_df["Target"] = temporal_robustness_df["metric"].map(
    TARGET_LABELS
)

if WRITE_INTERMEDIATE_OUTPUTS:
    atomic_write_parquet(
        temporal_robustness_df,
        TEMPORAL_ROBUSTNESS_PATH,
    )

temporal_display_df = (
    temporal_robustness_df[
        [
            "Target", "horizon", "month_count",
            "monthly_direction_agreement_pct", "monthly_sign_flips",
            "monthly_gap_pct_min", "monthly_gap_pct_max",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "month_count": "Months",
        "monthly_direction_agreement_pct": "Direction agreement %",
        "monthly_sign_flips": "Sign flips",
        "monthly_gap_pct_min": "Monthly min %",
        "monthly_gap_pct_max": "Monthly max %",
    })
)

display(temporal_display_df.round(3))

,Target,Horizon,Months,Direction agreement %,Sign flips,Monthly min %,Monthly max %
0,FHVHV average speed,1,15,100.000,0,2.607,12.734
1,FHVHV average speed,2,15,100.000,0,2.843,12.493
2,FHVHV average speed,5,15,100.000,0,3.722,15.302
3,FHVHV trips,1,15,100.000,0,-15.401,-6.313
4,FHVHV trips,2,15,100.000,0,-14.617,-6.645
5,FHVHV trips,5,15,100.000,0,-14.617,-6.645
6,Subway ridership,1,15,100.000,0,-22.997,-8.672
7,Subway ridership,2,15,100.000,0,-21.605,-8.015
8,Subway ridership,5,15,100.000,0,-21.605,-8.015
9,Taxi average speed,1,15,60.000,7,-2.597,4.900


**Findings.** Almost every result keeps the same direction throughout the 15-month post-congestion-pricing period. The one clear exception is **Taxi average speed at h=1**: only 60% of months match its overall positive result, and the monthly gaps range from **−2.6% to +4.9%**. The other 14 Target × Horizon results keep the same direction in every month, so Taxi speed h=1 is the main result that is not consistently visible over time.

**5.3.1.4B — Check whether the headline direction is broad across larger mobility geographies.**

The final 5.2.1 spatial handoff already contains borough, policy-geography, and Pre-CP mobility-environment results. For each Target × Horizon, measure how often those broader groups point in the same direction as the system-wide primary result.

In [9]:
# ---------------------------------------------------------------------
# 5.3.1.4B — Summarize geographic consistency from the 5.2.1 final table
# ---------------------------------------------------------------------

require_upstream_handoff()

# Use the saved system-wide primary results when the in-memory copy is absent.
if "population_robustness_df" not in globals():
    population_robustness_df = load_saved_table(
        POPULATION_ROBUSTNESS_PATH,
        "5.3.1 population robustness",
        ["metric", "horizon", "primary_gap"],
    )

# Broader geography lenses show whether the system-wide direction is repeated
# across boroughs, policy geography, and Pre-CP mobility environments.
GEOGRAPHY_GRAINS = {
    "borough": "Borough",
    "policy_geography": "Policy geography",
    "pre_cp_mobility_environment": "Pre-CP mobility environment",
}

available_geography_grains = set(
    spatial_summary_df["summary_grain"].dropna().astype(str)
)
missing_geography_grains = sorted(
    set(GEOGRAPHY_GRAINS) - available_geography_grains
)

if missing_geography_grains:
    raise RuntimeError(
        "The validated 5.2.1 spatial handoff is missing expected grains: "
        f"{missing_geography_grains}. "
        f"Available grains: {sorted(available_geography_grains)}"
    )

geography_robustness_rows = []

for overall_row in population_robustness_df.itertuples(index=False):
    metric = overall_row.metric
    horizon = int(overall_row.horizon)
    overall_sign = np.sign(overall_row.primary_gap)

    for summary_grain, geography_label in GEOGRAPHY_GRAINS.items():
        group_df = spatial_summary_df.loc[
            spatial_summary_df["summary_grain"].eq(summary_grain)
            & spatial_summary_df["metric"].eq(metric)
            & spatial_summary_df["horizon"].eq(horizon)
            & spatial_summary_df["counterfactual_gap"].notna()
        ].copy()

        if group_df.empty:
            raise RuntimeError(
                f"{TARGET_LABELS[metric]} h={horizon}: "
                f"no `{summary_grain}` rows were found."
            )

        group_signs = np.sign(
            group_df["counterfactual_gap"].to_numpy(dtype=float)
        )

        # Direction agreement is the share of broader groups that point the
        # same way as the system-wide primary result.
        direction_agreement_pct = (
            100 * np.mean(group_signs == overall_sign)
            if overall_sign != 0
            else np.nan
        )

        geography_robustness_rows.append({
            "metric": metric,
            "horizon": horizon,
            "geography_dimension": geography_label,
            "group_count": len(group_df),
            "direction_agreement_pct": direction_agreement_pct,
            "gap_pct_min": group_df["counterfactual_gap_pct"].min(),
            "gap_pct_max": group_df["counterfactual_gap_pct"].max(),
        })

geography_robustness_df = (
    pd.DataFrame(geography_robustness_rows)
    .sort_values(["metric", "horizon", "geography_dimension"])
    .reset_index(drop=True)
)

expected_geography_rows = EXPECTED_JOB_COUNT * len(GEOGRAPHY_GRAINS)

if len(geography_robustness_df) != expected_geography_rows:
    raise RuntimeError(
        "Geographic robustness produced "
        f"{len(geography_robustness_df)} rows; "
        f"expected {expected_geography_rows}."
    )

if geography_robustness_df[
    ["metric", "horizon", "geography_dimension"]
].duplicated().any():
    raise RuntimeError(
        "Geographic robustness contains duplicate "
        "Target × Horizon × Geography rows."
    )

geography_robustness_df["Target"] = geography_robustness_df["metric"].map(
    TARGET_LABELS
)

if WRITE_INTERMEDIATE_OUTPUTS:
    atomic_write_parquet(
        geography_robustness_df,
        GEOGRAPHY_ROBUSTNESS_PATH,
    )

# Pivot the three geography lenses onto one row per Target × Horizon.
geography_display_df = (
    geography_robustness_df.pivot(
        index=["Target", "horizon"],
        columns="geography_dimension",
        values="direction_agreement_pct",
    )
    .reset_index()
    .rename(columns={"horizon": "Horizon"})
)
geography_display_df.columns.name = None

display(geography_display_df.round(3))

,Target,Horizon,Borough,Policy geography,Pre-CP mobility environment
0,FHVHV average speed,1,100.000,100.000,100.000
1,FHVHV average speed,2,100.000,100.000,100.000
2,FHVHV average speed,5,100.000,100.000,100.000
3,FHVHV trips,1,83.333,80.000,100.000
4,FHVHV trips,2,83.333,80.000,100.000
5,FHVHV trips,5,83.333,80.000,100.000
6,Subway ridership,1,100.000,100.000,100.000
7,Subway ridership,2,100.000,100.000,100.000
8,Subway ridership,5,100.000,100.000,100.000
9,Taxi average speed,1,40.000,80.000,75.000


**Findings.** Most results also appear across different parts of the city rather than being concentrated in one type of geography. Subway ridership and FHVHV speed keep the same direction across every borough, policy-geography group, and mobility environment we tested. The weakest result is again **Taxi average speed at h=1**, where only 40% of borough groups match the citywide direction. Taxi trips are also somewhat uneven across geography, although their overall negative direction remains broad.

**5.3.1.4C — Summarize cross-horizon consistency on exact common support.**

The three horizons are different forecast distances. Use the aligned 5.2.1 horizon-stability surface to summarize whether h=1, h=2, and h=5 tell the same directional story when all three are compared on the same future observations.

In [10]:
# ---------------------------------------------------------------------
# 5.3.1.4C — Summarize exact-common-support horizon robustness
# ---------------------------------------------------------------------

require_upstream_handoff()

horizon_columns = [
    "metric",
    "primary_cross_regime",
    "target_observed_value",
    "target_observed_available",
    "observed_activity_weight",
    "all_horizons_available",
    "no_cp_h1", "no_cp_h2", "no_cp_h5",
    "no_cp_activity_weight_h1",
    "no_cp_activity_weight_h2",
    "no_cp_activity_weight_h5",
    "horizon_spread_pct",
    "direction_agreement_eligible",
    "horizon_direction_agreement",
]

horizon_gap_rows = []
horizon_metric_rows = []

for metric in FORECAST_TARGETS:
    # Read aligned three-horizon rows one target at a time to keep memory use modest.
    metric_df = pd.read_parquet(
        HORIZON_STABILITY_SURFACE_PATH,
        columns=horizon_columns,
        filters=[("metric", "==", metric)],
    )
    require_columns(
        metric_df,
        horizon_columns,
        f"{TARGET_LABELS[metric]} horizon-stability rows",
    )

    common_mask = (
        metric_df["primary_cross_regime"].astype(bool)
        & metric_df["all_horizons_available"].astype(bool)
        & metric_df["target_observed_available"].astype(bool)
        & metric_df["target_observed_value"].notna()
    )
    common_df = metric_df.loc[common_mask].copy()

    if metric in SPEED_TARGETS:
        weight_columns = [
            "observed_activity_weight",
            "no_cp_activity_weight_h1",
            "no_cp_activity_weight_h2",
            "no_cp_activity_weight_h5",
        ]

        # Exact common support requires all four activity weights on the same rows.
        common_df = common_df.loc[
            common_df[weight_columns].notna().all(axis=1)
        ].copy()

    if common_df.empty:
        raise RuntimeError(
            f"{TARGET_LABELS[metric]} has no exact three-horizon common support."
        )

    if metric in SPEED_TARGETS:
        observed_denominator = common_df["observed_activity_weight"].sum()
        observed_level = (
            (
                common_df["target_observed_value"]
                * common_df["observed_activity_weight"]
            ).sum()
            / observed_denominator
            if observed_denominator > 0
            else np.nan
        )
    else:
        observed_level = float(common_df["target_observed_value"].sum())

    metric_gap_pcts = {}

    for horizon in FORECAST_HORIZONS:
        forecast_column = f"no_cp_h{horizon}"

        if metric in SPEED_TARGETS:
            weight_column = f"no_cp_activity_weight_h{horizon}"
            no_cp_denominator = common_df[weight_column].sum()
            no_cp_level = (
                (
                    common_df[forecast_column]
                    * common_df[weight_column]
                ).sum()
                / no_cp_denominator
                if no_cp_denominator > 0
                else np.nan
            )
        else:
            no_cp_level = float(common_df[forecast_column].sum())

        counterfactual_gap = no_cp_level - observed_level
        counterfactual_gap_pct = (
            100 * counterfactual_gap / observed_level
            if observed_level != 0
            else np.nan
        )
        metric_gap_pcts[horizon] = counterfactual_gap_pct

        horizon_gap_rows.append({
            "metric": metric,
            "horizon": horizon,
            "common_support_rows": len(common_df),
            "observed_level": observed_level,
            "no_cp_level": no_cp_level,
            "counterfactual_gap": counterfactual_gap,
            "counterfactual_gap_pct": counterfactual_gap_pct,
        })

    # Pair the full-period comparison with row-level horizon-agreement evidence.
    direction_df = metric_df.loc[
        metric_df["primary_cross_regime"].astype(bool)
        & metric_df["direction_agreement_eligible"].astype(bool)
        & metric_df["horizon_direction_agreement"].notna()
    ].copy()

    if direction_df.empty:
        raise RuntimeError(
            f"{TARGET_LABELS[metric]} has no eligible row-level horizon evidence."
        )

    gap_pct_values = np.array(
        [metric_gap_pcts[h] for h in FORECAST_HORIZONS],
        dtype=float,
    )
    aggregate_signs = np.sign(gap_pct_values)

    horizon_metric_rows.append({
        "metric": metric,
        "h1_gap_pct": metric_gap_pcts[1],
        "h2_gap_pct": metric_gap_pcts[2],
        "h5_gap_pct": metric_gap_pcts[5],
        "aggregate_horizons_same_direction":
            len(set(aggregate_signs.tolist())) == 1,
        "aggregate_horizon_spread_pp":
            float(np.nanmax(gap_pct_values) - np.nanmin(gap_pct_values)),
        "row_direction_agreement_pct":
            100 * direction_df["horizon_direction_agreement"].astype(bool).mean(),
        "median_row_horizon_spread_pct":
            direction_df["horizon_spread_pct"].median(),
    })

    del metric_df, common_df, direction_df
    gc.collect()

horizon_common_support_df = pd.DataFrame(horizon_gap_rows)
horizon_robustness_df = (
    pd.DataFrame(horizon_metric_rows)
    .sort_values("metric")
    .reset_index(drop=True)
)

if len(horizon_common_support_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Exact-common-support horizon analysis did not produce all 15 jobs."
    )

if len(horizon_robustness_df) != len(FORECAST_TARGETS):
    raise RuntimeError(
        "Horizon robustness did not produce one summary per target."
    )

horizon_robustness_df["Target"] = horizon_robustness_df["metric"].map(
    TARGET_LABELS
)

if WRITE_INTERMEDIATE_OUTPUTS:
    atomic_write_parquet(
        horizon_robustness_df,
        HORIZON_ROBUSTNESS_PATH,
    )

horizon_display_df = (
    horizon_robustness_df[
        [
            "Target",
            "h1_gap_pct", "h2_gap_pct", "h5_gap_pct",
            "aggregate_horizon_spread_pp",
            "aggregate_horizons_same_direction",
            "row_direction_agreement_pct",
            "median_row_horizon_spread_pct",
        ]
    ]
    .rename(columns={
        "h1_gap_pct": "h=1 gap %",
        "h2_gap_pct": "h=2 gap %",
        "h5_gap_pct": "h=5 gap %",
        "aggregate_horizon_spread_pp": "Aggregate spread (pp)",
        "aggregate_horizons_same_direction": "Aggregate same direction",
        "row_direction_agreement_pct": "Row direction agreement %",
        "median_row_horizon_spread_pct": "Median row spread %",
    })
)

display(horizon_display_df.round(3))


# ---------------------------------------------------------------------
# Visual check — the 5 Target × 3 Horizon counterfactual overview
# ---------------------------------------------------------------------

horizon_heatmap_df = (
    horizon_common_support_df.pivot(
        index="metric",
        columns="horizon",
        values="counterfactual_gap_pct",
    )
    .reindex(index=FORECAST_TARGETS, columns=FORECAST_HORIZONS)
)
horizon_heatmap_values = horizon_heatmap_df.to_numpy(dtype=float)
horizon_color_limit = np.nanmax(np.abs(horizon_heatmap_values))

if not np.isfinite(horizon_color_limit) or horizon_color_limit == 0:
    horizon_color_limit = 1.0

horizon_fig = go.Figure(
    go.Heatmap(
        z=horizon_heatmap_values,
        x=[HORIZON_LABELS[h] for h in FORECAST_HORIZONS],
        y=[TARGET_LABELS[m] for m in FORECAST_TARGETS],
        colorscale=BRAND_DIVERGING_SEQUENCE,
        zmin=-horizon_color_limit,
        zmax=horizon_color_limit,
        zmid=0,
        colorbar={"title": "Gap<br>(%)", "tickformat": ".3f"},
        text=np.round(horizon_heatmap_values, 3),
        texttemplate="%{text:+.3f}%",
        hovertemplate=(
            "<b>%{y}</b><br>%{x}<br>"
            "No-CP − observed: %{z:+.3f}%<extra></extra>"
        ),
    )
)

apply_branding(horizon_fig)
horizon_fig.update_layout(
    title={
        "text": (
            "<b>Do h=1, h=2, and h=5 Tell the Same Full-Period Story?</b>"
            "<br><sup>Primary population · exact three-horizon common support</sup>"
        ),
        "x": 0.5,
    },
    width=820,
    height=470,
    margin={"l": 150, "r": 70, "t": 100, "b": 55},
    xaxis={"title": "Forecast horizon"},
    yaxis={"title": ""},
)

if RUN_VISUALS:
    horizon_fig.show()

,Target,h=1 gap %,h=2 gap %,h=5 gap %,Aggregate spread (pp),Aggregate same direction,Row direction agreement %,Median row spread %
0,FHVHV average speed,6.212,6.272,6.866,0.654,True,85.167,3.418
1,FHVHV trips,-11.004,-11.028,-11.028,0.025,True,90.808,3.351
2,Subway ridership,-14.276,-14.443,-14.443,0.167,True,95.447,4.198
3,Taxi average speed,1.322,12.213,5.399,10.891,True,72.698,13.063
4,Taxi trips,-28.346,-28.441,-28.197,0.245,True,84.348,36.710


**Findings.** All five mobility measures keep the same overall direction at h=1, h=2, and h=5, but the size of the result can change with forecast horizon. **FHVHV trips and Subway ridership are especially consistent**, with very similar results across all three horizons. **Taxi average speed changes the most**, rising from about **+1.3% at h=1** to **+12.2% at h=2** and **+5.4% at h=5**. Taxi trips stay close to **−28%** at all three horizons overall, even though individual Taxi Zone × time observations can vary much more.

## 5.3.1.5 Put the Post-CP Gaps on the Pre-CP Forecast-Error Scale

A raw gap in trips, riders, or miles per hour cannot be compared fairly across mobility measures. 5.2.1 expressed each supported gap in units of that Target × Horizon model's frozen **Pre-CP validation MAE**, giving all five targets one common forecast-error scale.

One MAE unit means the post-CP difference is as large as that forecasting job's average absolute Pre-CP validation error. This section summarizes that scale across all 15 Target × Horizon results so the size of the observed-versus-no-CP divergence can be compared across mobility measures.

**5.3.1.5A — Summarize post-CP gaps in Pre-CP validation-MAE units.**

For every Target × Horizon, summarize the typical absolute gap, the larger end of the distribution, and the share of rows above one or two MAE units. These descriptive reference points show how large the post-CP differences are relative to each forecasting job's ordinary Pre-CP error.

In [11]:
# ---------------------------------------------------------------------
# 5.3.1.5A — Put post-CP gaps on the frozen Pre-CP MAE scale
# ---------------------------------------------------------------------

require_upstream_handoff()

calibration_columns = [
    "horizon",
    "primary_gap_eligible",
    "counterfactual_gap_mae_units",
]
calibration_rows = []

for metric in FORECAST_TARGETS:
    # Use the frozen standardized gap so every comparison keeps the same
    # Pre-CP validation-MAE denominator.
    metric_df = read_gap_metric(metric, calibration_columns)
    require_columns(
        metric_df,
        ["metric", *calibration_columns],
        f"{TARGET_LABELS[metric]} MAE-calibration rows",
    )
    metric_df["horizon"] = metric_df["horizon"].astype(int)

    # Calibration describes the size of the gap within the primary supported population.
    metric_df = metric_df.loc[
        metric_df["primary_gap_eligible"].fillna(False).astype(bool)
    ].copy()

    metric_df["counterfactual_gap_mae_units"] = pd.to_numeric(
        metric_df["counterfactual_gap_mae_units"],
        errors="coerce",
    )
    metric_df = metric_df.loc[
        np.isfinite(metric_df["counterfactual_gap_mae_units"])
    ].copy()
    metric_df["absolute_mae_units"] = metric_df[
        "counterfactual_gap_mae_units"
    ].abs()

    for horizon in FORECAST_HORIZONS:
        horizon_df = metric_df.loc[metric_df["horizon"].eq(horizon)]
        absolute_values = horizon_df["absolute_mae_units"]

        if horizon_df.empty:
            raise RuntimeError(
                f"{TARGET_LABELS[metric]} h={horizon}: "
                "no supported MAE-calibration rows."
            )

        # Median captures a typical row; P90 keeps the upper tail visible.
        calibration_rows.append({
            "metric": metric,
            "horizon": horizon,
            "rows": len(horizon_df),
            "median_signed_mae_units":
                horizon_df["counterfactual_gap_mae_units"].median(),
            "median_absolute_mae_units": absolute_values.median(),
            "p90_absolute_mae_units": absolute_values.quantile(0.90),
            "share_abs_ge_1_mae_pct": 100 * absolute_values.ge(1).mean(),
            "share_abs_ge_2_mae_pct": 100 * absolute_values.ge(2).mean(),
        })

    del metric_df
    gc.collect()

pre_cp_error_calibration_df = (
    pd.DataFrame(calibration_rows)
    .sort_values(["metric", "horizon"])
    .reset_index(drop=True)
)

if len(pre_cp_error_calibration_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Pre-CP error calibration did not produce all 15 Target × Horizon jobs."
    )

if pre_cp_error_calibration_df[["metric", "horizon"]].duplicated().any():
    raise RuntimeError(
        "Pre-CP error calibration contains duplicate Target × Horizon jobs."
    )

pre_cp_error_calibration_df["Target"] = pre_cp_error_calibration_df[
    "metric"
].map(TARGET_LABELS)

if WRITE_INTERMEDIATE_OUTPUTS:
    atomic_write_parquet(
        pre_cp_error_calibration_df,
        PRE_CP_CALIBRATION_PATH,
    )

calibration_display_df = (
    pre_cp_error_calibration_df[
        [
            "Target", "horizon", "rows",
            "median_signed_mae_units",
            "median_absolute_mae_units",
            "p90_absolute_mae_units",
            "share_abs_ge_1_mae_pct",
            "share_abs_ge_2_mae_pct",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "rows": "Rows",
        "median_signed_mae_units": "Median signed MAE units",
        "median_absolute_mae_units": "Median |MAE units|",
        "p90_absolute_mae_units": "P90 |MAE units|",
        "share_abs_ge_1_mae_pct": "Rows ≥1 MAE unit %",
        "share_abs_ge_2_mae_pct": "Rows ≥2 MAE units %",
    })
)

display(calibration_display_df.round(3))


# ---------------------------------------------------------------------
# Visual check — compare unlike mobility targets on one error scale
# ---------------------------------------------------------------------

calibration_heatmap_df = (
    pre_cp_error_calibration_df.pivot(
        index="metric",
        columns="horizon",
        values="median_absolute_mae_units",
    )
    .reindex(index=FORECAST_TARGETS, columns=FORECAST_HORIZONS)
)
calibration_heatmap_values = calibration_heatmap_df.to_numpy(dtype=float)

calibration_fig = go.Figure(
    go.Heatmap(
        z=calibration_heatmap_values,
        x=[HORIZON_LABELS[h] for h in FORECAST_HORIZONS],
        y=[TARGET_LABELS[m] for m in FORECAST_TARGETS],
        colorscale=BRAND_SEQUENTIAL_HEATMAP_SCALE,
        colorbar={"title": "|MAE<br>units|", "tickformat": ".3f"},
        text=np.round(calibration_heatmap_values, 3),
        texttemplate="%{text:.3f}",
        hovertemplate=(
            "<b>%{y}</b><br>%{x}<br>"
            "Median absolute gap: %{z:.3f} Pre-CP MAE units"
            "<extra></extra>"
        ),
    )
)

apply_branding(calibration_fig)
calibration_fig.update_layout(
    title={
        "text": (
            "<b>How Large Are the Post-CP Gaps Relative to Pre-CP Forecast Error?</b>"
            "<br><sup>Median absolute row-level gap · "
            "primary cross-regime population</sup>"
        ),
        "x": 0.5,
    },
    width=840,
    height=475,
    margin={"l": 150, "r": 85, "t": 105, "b": 55},
    xaxis={"title": "Forecast horizon"},
    yaxis={"title": ""},
)

if RUN_VISUALS:
    calibration_fig.show()

,Target,Horizon,Rows,Median signed MAE units,Median |MAE units|,P90 |MAE units|,Rows ≥1 MAE unit %,Rows ≥2 MAE units %
0,FHVHV average speed,1,576386,0.766,1.109,2.937,54.160,24.026
1,FHVHV average speed,2,576113,0.742,1.098,2.941,53.776,23.763
2,FHVHV average speed,5,575366,0.800,1.086,2.926,53.383,23.203
3,FHVHV trips,1,578785,-0.385,0.786,4.089,43.761,25.297
4,FHVHV trips,2,578512,-0.372,0.784,4.099,43.733,25.295
5,FHVHV trips,5,577745,-0.373,0.785,4.105,43.782,25.331
6,Subway ridership,1,334434,-0.597,0.931,6.269,47.977,29.992
7,Subway ridership,2,334280,-0.564,0.883,5.934,46.589,28.861
8,Subway ridership,5,333835,-0.564,0.883,5.930,46.591,28.860
9,Taxi average speed,1,247181,0.061,0.610,1.894,29.688,8.887


**Findings.** The counterfactual gaps are not equally large compared with the forecasting system’s normal errors. A typical FHVHV-speed gap is a little larger than one ordinary Pre-CP forecasting error, while Subway ridership, FHVHV trips, and Taxi speed are generally around one error or less. Taxi trips are only about **one-quarter of a normal forecasting error for the typical row**, even though their citywide percentage gap is much larger. This is an important reminder that a large citywide percentage difference does not necessarily mean that every individual place and time is unusually far from what the model normally misses by.

**5.3.1.5B — Bring the completed robustness evidence into one review table.**

The Decision Point should make the evidence easy to compare across all 15 Target × Horizon results. This table places the key robustness measures on one row for each result so the final interpretation can consider the evidence together.

In [12]:
# ---------------------------------------------------------------------
# 5.3.1.5B — Assemble the pre-Decision robustness evidence matrix
# ---------------------------------------------------------------------

require_upstream_handoff()


# ---------------------------------------------------------------------
# Load compact robustness evidence when needed for restart
# ---------------------------------------------------------------------

if "population_robustness_df" not in globals():
    population_robustness_df = load_saved_table(
        POPULATION_ROBUSTNESS_PATH,
        "5.3.1 population robustness",
        [
            "metric", "horizon",
            "primary_gap_pct",
            "same_direction",
            "gap_pct_shift_pp",
        ],
    )

if "speed_weighting_robustness_df" not in globals():
    speed_weighting_robustness_df = load_saved_table(
        SPEED_WEIGHTING_ROBUSTNESS_PATH,
        "5.3.1 speed-weighting robustness",
        ["metric", "horizon", "same_direction", "gap_pct_shift_pp"],
    )

if "temporal_robustness_df" not in globals():
    temporal_robustness_df = load_saved_table(
        TEMPORAL_ROBUSTNESS_PATH,
        "5.3.1 temporal robustness",
        [
            "metric", "horizon",
            "monthly_direction_agreement_pct",
            "monthly_sign_flips",
        ],
    )

if "geography_robustness_df" not in globals():
    geography_robustness_df = load_saved_table(
        GEOGRAPHY_ROBUSTNESS_PATH,
        "5.3.1 geography robustness",
        ["metric", "horizon", "direction_agreement_pct"],
    )

if "horizon_robustness_df" not in globals():
    horizon_robustness_df = load_saved_table(
        HORIZON_ROBUSTNESS_PATH,
        "5.3.1 horizon robustness",
        [
            "metric",
            "aggregate_horizons_same_direction",
            "aggregate_horizon_spread_pp",
            "row_direction_agreement_pct",
        ],
    )

if "pre_cp_error_calibration_df" not in globals():
    pre_cp_error_calibration_df = load_saved_table(
        PRE_CP_CALIBRATION_PATH,
        "5.3.1 Pre-CP error calibration",
        [
            "metric", "horizon",
            "median_absolute_mae_units",
            "p90_absolute_mae_units",
        ],
    )


# Geography contributes three lenses per job. Preserve both the weakest
# agreement and the average so differences among those lenses stay visible.
geography_job_summary_df = (
    geography_robustness_df.groupby(
        ["metric", "horizon"],
        observed=True,
    )
    .agg(
        geography_min_direction_agreement_pct=(
            "direction_agreement_pct",
            "min",
        ),
        geography_mean_direction_agreement_pct=(
            "direction_agreement_pct",
            "mean",
        ),
    )
    .reset_index()
)

# Weighting sensitivity applies only to the two speed targets; the remaining
# jobs therefore keep these fields missing by design.
speed_evidence_df = (
    speed_weighting_robustness_df[
        ["metric", "horizon", "same_direction", "gap_pct_shift_pp"]
    ]
    .rename(columns={
        "same_direction": "speed_weight_same_direction",
        "gap_pct_shift_pp": "speed_weight_gap_shift_pp",
    })
)


# Join every evidence dimension onto the frozen 15 Target × Horizon jobs.
predecision_evidence_df = (
    population_robustness_df[
        [
            "metric", "horizon",
            "primary_gap_pct", "same_direction", "gap_pct_shift_pp",
        ]
    ]
    .rename(columns={
        "same_direction": "population_same_direction",
        "gap_pct_shift_pp": "population_gap_shift_pp",
    })
    .merge(
        speed_evidence_df,
        on=["metric", "horizon"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        temporal_robustness_df[
            [
                "metric", "horizon",
                "monthly_direction_agreement_pct",
                "monthly_sign_flips",
            ]
        ],
        on=["metric", "horizon"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        geography_job_summary_df,
        on=["metric", "horizon"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        horizon_robustness_df[
            [
                "metric",
                "aggregate_horizons_same_direction",
                "aggregate_horizon_spread_pp",
                "row_direction_agreement_pct",
            ]
        ],
        on="metric",
        how="left",
        validate="many_to_one",
    )
    .merge(
        pre_cp_error_calibration_df[
            [
                "metric", "horizon",
                "median_absolute_mae_units",
                "p90_absolute_mae_units",
            ]
        ],
        on=["metric", "horizon"],
        how="left",
        validate="one_to_one",
    )
)

predecision_evidence_df["Target"] = predecision_evidence_df["metric"].map(
    TARGET_LABELS
)
predecision_evidence_df = (
    predecision_evidence_df
    .sort_values(["metric", "horizon"])
    .reset_index(drop=True)
)

# One row per Target × Horizon keeps the Decision Point aligned to the
# frozen 15-job forecasting contract.
if len(predecision_evidence_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "The pre-Decision evidence matrix does not contain all 15 "
        "Target × Horizon jobs."
    )

if predecision_evidence_df[["metric", "horizon"]].duplicated().any():
    raise RuntimeError(
        "The pre-Decision evidence matrix contains duplicate jobs."
    )

if WRITE_INTERMEDIATE_OUTPUTS:
    atomic_write_parquet(
        predecision_evidence_df,
        PREDECISION_EVIDENCE_PATH,
    )

predecision_display_df = (
    predecision_evidence_df[
        [
            "Target", "horizon",
            "primary_gap_pct",
            "population_same_direction",
            "population_gap_shift_pp",
            "speed_weight_same_direction",
            "speed_weight_gap_shift_pp",
            "monthly_direction_agreement_pct",
            "geography_min_direction_agreement_pct",
            "aggregate_horizons_same_direction",
            "row_direction_agreement_pct",
            "aggregate_horizon_spread_pp",
            "median_absolute_mae_units",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "primary_gap_pct": "Baseline gap %",
        "population_same_direction": "Population dir. agrees",
        "population_gap_shift_pp": "Population shift pp",
        "speed_weight_same_direction": "Weight dir. agrees",
        "speed_weight_gap_shift_pp": "Weight shift pp",
        "monthly_direction_agreement_pct": "Monthly dir. agreement %",
        "geography_min_direction_agreement_pct": "Lowest geography agreement %",
        "aggregate_horizons_same_direction": "Horizons agree overall",
        "row_direction_agreement_pct": "Row horizon agreement %",
        "aggregate_horizon_spread_pp": "Horizon spread pp",
        "median_absolute_mae_units": "Median |MAE units|",
    })
)

display(predecision_display_df.round(3))

,Target,Horizon,Baseline gap %,Population dir. agrees,Population shift pp,Weight dir. agrees,Weight shift pp,Monthly dir. agreement %,Lowest geography agreement %,Horizons agree overall,Row horizon agreement %,Horizon spread pp,Median |MAE units|
0,FHVHV average speed,1,6.200,True,0.001,True,-0.365,100.000,100.000,True,85.167,0.654,1.109
1,FHVHV average speed,2,6.257,True,0.001,True,-0.252,100.000,100.000,True,85.167,0.654,1.098
2,FHVHV average speed,5,6.866,True,0.001,True,-0.385,100.000,100.000,True,85.167,0.654,1.086
3,FHVHV trips,1,-10.997,True,0.004,NaN,NaN,100.000,80.000,True,90.808,0.025,0.786
4,FHVHV trips,2,-11.028,True,0.004,NaN,NaN,100.000,80.000,True,90.808,0.025,0.784
5,FHVHV trips,5,-11.028,True,0.004,NaN,NaN,100.000,80.000,True,90.808,0.025,0.785
6,Subway ridership,1,-14.231,True,-0.030,NaN,NaN,100.000,100.000,True,95.447,0.167,0.931
7,Subway ridership,2,-14.417,True,-0.030,NaN,NaN,100.000,100.000,True,95.447,0.167,0.883
8,Subway ridership,5,-14.443,True,-0.030,NaN,NaN,100.000,100.000,True,95.447,0.167,0.883
9,Taxi average speed,1,1.308,True,0.000,True,1.873,60.000,40.000,True,72.698,10.891,0.610


**Findings.** Most of the counterfactual results hold up well across the different checks. Changing the supported population barely matters, and 14 of the 15 Target × Horizon results keep the same direction in every post-CP month. Subway ridership and both FHVHV measures are especially consistent, while Taxi trips keep a stable overall direction despite more local variation. **Taxi average speed is the main result that needs caution**, especially h=1, because its result changes across months and boroughs and its size changes noticeably depending on forecast horizon and weighting.

> Decision — Freeze the final interpretation of the robustness evidence

The robustness evidence is now assembled for all 15 Target × Horizon results\. For each result, freeze four reader\-facing decisions: whether the evidence is stable, mixed, or sensitive; the main caution that should travel with it; the strongest supporting evidence view; and whether it should be included in the Showcase evidence surface\.

These decisions become the interpretation layer carried into the final Chapter 5 handoff and Showcase\.

### Section Summary

Most of the headline counterfactual results are not fragile. FHVHV trips, FHVHV speed, Subway ridership, and the overall Taxi-trip result remain consistent across the major checks we tested. **Taxi average speed is the exception:** h=1 is weak across both time and borough geography, while the size of the Taxi-speed result changes substantially across horizons and under an alternative weighting approach. These checks give us enough evidence to separate the final results into stable, mixed, and sensitive groups.

## 5.3.1.6 Build the Final Counterfactual Evidence Surfaces

This section combines the completed robustness evidence with the frozen Decision Point outcomes into two downstream surfaces: a complete 15-job robustness matrix for detailed inspection and a compact conclusion registry for reader-facing interpretation.

Together, they give downstream notebooks and the Showcase one consistent source for both the supporting evidence and its final interpretation.

**5.3.1.6A — Record the Decision Point outcomes.**

Record one final interpretation for every Target × Horizon result. Each row keeps the evidence status, the caution that should accompany the result, the strongest supporting view, and whether the result belongs in the Showcase evidence surface.

In [13]:
# ---------------------------------------------------------------------
# 5.3.1.6A — Record the Decision Point outcomes
# ---------------------------------------------------------------------

require_upstream_handoff()


# ---------------------------------------------------------------------
# Load the 15-job evidence surface
# ---------------------------------------------------------------------

if "predecision_evidence_df" not in globals():
    if not PREDECISION_EVIDENCE_PATH.exists():
        raise RuntimeError(
            "Run 5.3.1.5B before 5.3.1.6A."
        )

    predecision_evidence_df = pd.read_parquet(
        PREDECISION_EVIDENCE_PATH
    )

if len(predecision_evidence_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "The pre-Decision evidence does not contain all "
        f"{EXPECTED_JOB_COUNT} Target × Horizon jobs."
    )

if predecision_evidence_df[
    ["metric", "horizon"]
].duplicated().any():
    raise RuntimeError(
        "The pre-Decision evidence contains duplicate "
        "Target × Horizon jobs."
    )


# ---------------------------------------------------------------------
# Frozen interpretation
#
# Stable:
#   The main result remains consistent across the checks.
#
# Mixed:
#   The direction is reasonably consistent, but the size of the result
#   changes enough that readers should see that qualification.
#
# Sensitive:
#   Important checks disagree enough that the result needs clear caution.
#
# Each tuple stores:
#   evidence_status, caution_note, supporting_view, showcase_include
# ---------------------------------------------------------------------

DECISION_INPUTS = {
    # Taxi trips
    ("taxi_trip_count", 1): (
        "stable",
        (
            "The overall negative result is very consistent over time and "
            "across forecast horizons. Local percentage gaps vary more in "
            "places and times with very small observed trip counts."
        ),
        "Time, geography, and forecast-horizon checks",
        True,
    ),
    ("taxi_trip_count", 2): (
        "stable",
        (
            "The overall negative result is very consistent over time and "
            "across forecast horizons. Local percentage gaps vary more in "
            "places and times with very small observed trip counts."
        ),
        "Time, geography, and forecast-horizon checks",
        True,
    ),
    ("taxi_trip_count", 5): (
        "stable",
        (
            "The overall negative result is very consistent over time and "
            "across forecast horizons. Local percentage gaps vary more in "
            "places and times with very small observed trip counts."
        ),
        "Time, geography, and forecast-horizon checks",
        True,
    ),

    # Taxi average speed
    ("taxi_avg_trip_speed", 1): (
        "sensitive",
        (
            "The overall positive gap is small, only 60% of months match "
            "that direction, and only 40% of borough groups do. The size "
            "of the result also changes under alternative weighting and "
            "across forecast horizons."
        ),
        "Time, geography, weighting, and forecast-horizon checks",
        True,
    ),
    ("taxi_avg_trip_speed", 2): (
        "mixed",
        (
            "The positive direction is consistent within h=2, but the size "
            "of the Taxi-speed result changes noticeably under alternative "
            "weighting and across forecast horizons."
        ),
        "Weighting and forecast-horizon checks",
        True,
    ),
    ("taxi_avg_trip_speed", 5): (
        "mixed",
        (
            "The positive direction is consistent within h=5, but the size "
            "of the Taxi-speed result changes noticeably under alternative "
            "weighting and across forecast horizons."
        ),
        "Weighting and forecast-horizon checks",
        True,
    ),

    # FHVHV trips
    ("fhvhv_trip_count", 1): (
        "stable",
        (
            "The negative result remains consistent when we change the "
            "supported population and look across time, geography, and "
            "forecast horizon."
        ),
        "Population, time, geography, and forecast-horizon checks",
        True,
    ),
    ("fhvhv_trip_count", 2): (
        "stable",
        (
            "The negative result remains consistent when we change the "
            "supported population and look across time, geography, and "
            "forecast horizon."
        ),
        "Population, time, geography, and forecast-horizon checks",
        True,
    ),
    ("fhvhv_trip_count", 5): (
        "stable",
        (
            "The negative result remains consistent when we change the "
            "supported population and look across time, geography, and "
            "forecast horizon."
        ),
        "Population, time, geography, and forecast-horizon checks",
        True,
    ),

    # FHVHV average speed
    ("fhvhv_avg_trip_speed", 1): (
        "stable",
        (
            "The positive result remains consistent across the checks, and "
            "alternative trip weighting changes its size only modestly."
        ),
        "Weighting, time, geography, and forecast-horizon checks",
        True,
    ),
    ("fhvhv_avg_trip_speed", 2): (
        "stable",
        (
            "The positive result remains consistent across the checks, and "
            "alternative trip weighting changes its size only modestly."
        ),
        "Weighting, time, geography, and forecast-horizon checks",
        True,
    ),
    ("fhvhv_avg_trip_speed", 5): (
        "stable",
        (
            "The positive result remains consistent across the checks, and "
            "alternative trip weighting changes its size only modestly."
        ),
        "Weighting, time, geography, and forecast-horizon checks",
        True,
    ),

    # Subway ridership
    ("subway_ridership", 1): (
        "stable",
        (
            "The negative result is very consistent across months, broader "
            "geographies, and forecast horizons."
        ),
        "Time, geography, and forecast-horizon checks",
        True,
    ),
    ("subway_ridership", 2): (
        "stable",
        (
            "The negative result is very consistent across months, broader "
            "geographies, and forecast horizons."
        ),
        "Time, geography, and forecast-horizon checks",
        True,
    ),
    ("subway_ridership", 5): (
        "stable",
        (
            "The negative result is very consistent across months, broader "
            "geographies, and forecast horizons."
        ),
        "Time, geography, and forecast-horizon checks",
        True,
    ),
}

ALLOWED_EVIDENCE_STATUS = {
    "stable",
    "mixed",
    "sensitive",
}


# ---------------------------------------------------------------------
# Validate Decision Point coverage
# ---------------------------------------------------------------------

expected_decision_keys = {
    (metric, horizon)
    for metric in FORECAST_TARGETS
    for horizon in FORECAST_HORIZONS
}

observed_decision_keys = set(
    DECISION_INPUTS
)

if observed_decision_keys != expected_decision_keys:
    missing = sorted(
        expected_decision_keys - observed_decision_keys
    )
    extra = sorted(
        observed_decision_keys - expected_decision_keys
    )

    raise RuntimeError(
        "Decision registry does not match the frozen 15 jobs. "
        f"Missing={missing}; Extra={extra}"
    )


# ---------------------------------------------------------------------
# Build the decision registry
# ---------------------------------------------------------------------

decision_rows = []

for (metric, horizon), (
    evidence_status,
    caution_note,
    supporting_view,
    showcase_include,
) in DECISION_INPUTS.items():

    evidence_status = (
        str(evidence_status)
        .strip()
        .lower()
    )
    caution_note = str(
        caution_note
    ).strip()
    supporting_view = str(
        supporting_view
    ).strip()

    if evidence_status not in ALLOWED_EVIDENCE_STATUS:
        raise ValueError(
            f"{metric} h={horizon}: evidence_status must be one of "
            f"{sorted(ALLOWED_EVIDENCE_STATUS)}."
        )

    if not caution_note:
        raise ValueError(
            f"{metric} h={horizon}: caution_note cannot be blank."
        )

    if not supporting_view:
        raise ValueError(
            f"{metric} h={horizon}: supporting_view cannot be blank."
        )

    if not isinstance(showcase_include, bool):
        raise TypeError(
            f"{metric} h={horizon}: showcase_include must be True or False."
        )

    decision_rows.append({
        "metric": metric,
        "horizon": int(horizon),
        "evidence_status": evidence_status,
        "caution_note": caution_note,
        "supporting_view": supporting_view,
        "showcase_include": showcase_include,
    })


decision_registry_df = (
    pd.DataFrame(decision_rows)
    .sort_values(
        ["metric", "horizon"]
    )
    .reset_index(drop=True)
)

if len(decision_registry_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Decision registry does not contain all "
        f"{EXPECTED_JOB_COUNT} Target × Horizon jobs."
    )

if decision_registry_df[
    ["metric", "horizon"]
].duplicated().any():
    raise RuntimeError(
        "Decision registry contains duplicate Target × Horizon jobs."
    )


# ---------------------------------------------------------------------
# Attach the full-period baseline gap
# ---------------------------------------------------------------------

decision_registry_df = decision_registry_df.merge(
    predecision_evidence_df[
        [
            "metric",
            "horizon",
            "primary_gap_pct",
        ]
    ],
    on=["metric", "horizon"],
    how="left",
    validate="one_to_one",
)

if decision_registry_df["primary_gap_pct"].isna().any():
    missing_jobs = (
        decision_registry_df.loc[
            decision_registry_df["primary_gap_pct"].isna(),
            ["metric", "horizon"],
        ]
        .to_dict("records")
    )

    raise RuntimeError(
        f"Decision registry is missing baseline gaps for {missing_jobs}."
    )

decision_registry_df["Target"] = (
    decision_registry_df["metric"].map(
        TARGET_LABELS
    )
)


def baseline_result_statement(row):
    """
    Describe the full-period observed-versus-no-CP gap in plain language.
    """
    metric = row["metric"]
    horizon = int(row["horizon"])
    gap_pct = float(row["primary_gap_pct"])
    target_label = TARGET_LABELS[metric].lower()

    if np.isclose(gap_pct, 0):
        return (
            f"h={horizon}: the no-CP forecast and observed "
            f"{target_label} level were nearly equal over the "
            "full post-CP period."
        )

    direction = (
        "above"
        if gap_pct > 0
        else "below"
    )

    return (
        f"h={horizon}: the no-CP forecast was "
        f"{abs(gap_pct):.1f}% {direction} the observed "
        f"{target_label} level over the full post-CP period."
    )


decision_registry_df[
    "baseline_result_summary"
] = decision_registry_df.apply(
    baseline_result_statement,
    axis=1,
)

decision_registry_df = decision_registry_df[
    [
        "metric",
        "Target",
        "horizon",
        "evidence_status",
        "primary_gap_pct",
        "baseline_result_summary",
        "caution_note",
        "supporting_view",
        "showcase_include",
    ]
]


# ---------------------------------------------------------------------
# Summarize the frozen interpretation
# ---------------------------------------------------------------------

decision_status_summary_df = (
    decision_registry_df.groupby(
        "evidence_status",
        observed=True,
    )
    .size()
    .reindex(
        [
            "stable",
            "mixed",
            "sensitive",
        ],
        fill_value=0,
    )
    .rename("Jobs")
    .reset_index()
    .rename(
        columns={
            "evidence_status": "Evidence status",
        }
    )
)

decision_display_df = (
    decision_registry_df.rename(
        columns={
            "horizon": "Horizon",
            "evidence_status": "Evidence status",
            "primary_gap_pct": "Baseline gap %",
            "baseline_result_summary": "Baseline result",
            "caution_note": "Caution",
            "supporting_view": "Supporting view",
            "showcase_include": "Showcase",
        }
    )
)

display(
    decision_status_summary_df
)

display(
    decision_display_df.round(3)
)

,Evidence status,Jobs
0,stable,12
1,mixed,2
2,sensitive,1


,metric,Target,Horizon,Evidence status,Baseline gap %,Baseline result,Caution,Supporting view,Showcase
0,fhvhv_avg_trip_speed,FHVHV average speed,1,stable,6.200,h=1: the no-CP forecast was 6.2% above the observed fhvhv average speed level over the full post-CP period.,"The positive result remains consistent across the checks, and alternative trip weighting changes its size only modes...","Weighting, time, geography, and forecast-horizon checks",True
1,fhvhv_avg_trip_speed,FHVHV average speed,2,stable,6.257,h=2: the no-CP forecast was 6.3% above the observed fhvhv average speed level over the full post-CP period.,"The positive result remains consistent across the checks, and alternative trip weighting changes its size only modes...","Weighting, time, geography, and forecast-horizon checks",True
2,fhvhv_avg_trip_speed,FHVHV average speed,5,stable,6.866,h=5: the no-CP forecast was 6.9% above the observed fhvhv average speed level over the full post-CP period.,"The positive result remains consistent across the checks, and alternative trip weighting changes its size only modes...","Weighting, time, geography, and forecast-horizon checks",True
3,fhvhv_trip_count,FHVHV trips,1,stable,-10.997,h=1: the no-CP forecast was 11.0% below the observed fhvhv trips level over the full post-CP period.,"The negative result remains consistent when we change the supported population and look across time, geography, and ...","Population, time, geography, and forecast-horizon checks",True
4,fhvhv_trip_count,FHVHV trips,2,stable,-11.028,h=2: the no-CP forecast was 11.0% below the observed fhvhv trips level over the full post-CP period.,"The negative result remains consistent when we change the supported population and look across time, geography, and ...","Population, time, geography, and forecast-horizon checks",True
5,fhvhv_trip_count,FHVHV trips,5,stable,-11.028,h=5: the no-CP forecast was 11.0% below the observed fhvhv trips level over the full post-CP period.,"The negative result remains consistent when we change the supported population and look across time, geography, and ...","Population, time, geography, and forecast-horizon checks",True
6,subway_ridership,Subway ridership,1,stable,-14.231,h=1: the no-CP forecast was 14.2% below the observed subway ridership level over the full post-CP period.,"The negative result is very consistent across months, broader geographies, and forecast horizons.","Time, geography, and forecast-horizon checks",True
7,subway_ridership,Subway ridership,2,stable,-14.417,h=2: the no-CP forecast was 14.4% below the observed subway ridership level over the full post-CP period.,"The negative result is very consistent across months, broader geographies, and forecast horizons.","Time, geography, and forecast-horizon checks",True
8,subway_ridership,Subway ridership,5,stable,-14.443,h=5: the no-CP forecast was 14.4% below the observed subway ridership level over the full post-CP period.,"The negative result is very consistent across months, broader geographies, and forecast horizons.","Time, geography, and forecast-horizon checks",True
9,taxi_avg_trip_speed,Taxi average speed,1,sensitive,1.308,h=1: the no-CP forecast was 1.3% above the observed taxi average speed level over the full post-CP period.,"The overall positive gap is small, only 60% of months match that direction, and only 40% of borough groups do. The s...","Time, geography, weighting, and forecast-horizon checks",True


**Findings.** The final review classifies **12 of the 15 Target × Horizon results as stable, two as mixed, and one as sensitive**. All Taxi-trip, FHVHV-trip, FHVHV-speed, and Subway-ridership results are stable. The three Taxi-speed results carry the qualifications: **h=1 is sensitive**, while **h=2 and h=5 are mixed** because their directions remain consistent but the size of the result changes more across horizons and weighting choices. All 15 results remain available in the Showcase so readers can see those differences in evidence strength rather than having weaker results hidden.

**5.3.1.6B — Build the final robustness matrix and conclusion registry.**

The detailed matrix keeps the numerical evidence needed to inspect why a result received its final interpretation. The conclusion registry is intentionally smaller: it carries the frozen result statement, evidence status, caution, supporting view, and Showcase decision that downstream pages can present directly.


In [14]:
# ---------------------------------------------------------------------
# 5.3.1.6B — Build the final evidence surfaces
# ---------------------------------------------------------------------

require_upstream_handoff()

if "decision_registry_df" not in globals():
    raise RuntimeError(
        "Run 5.3.1.6A and complete the Decision Point before "
        "building the final evidence surfaces."
    )


# ---------------------------------------------------------------------
# Load compact evidence when needed for restart
# ---------------------------------------------------------------------

if "population_robustness_df" not in globals():
    population_robustness_df = load_saved_table(
        POPULATION_ROBUSTNESS_PATH,
        "5.3.1 population robustness",
        [
            "metric", "horizon",
            "native_gap_pct",
            "absolute_gap_pct_shift_pp",
        ],
    )

if "speed_weighting_robustness_df" not in globals():
    speed_weighting_robustness_df = load_saved_table(
        SPEED_WEIGHTING_ROBUSTNESS_PATH,
        "5.3.1 speed-weighting robustness",
        [
            "metric", "horizon",
            "common_observed_gap_pct",
            "gap_pct_shift_pp",
        ],
    )

if "temporal_robustness_df" not in globals():
    temporal_robustness_df = load_saved_table(
        TEMPORAL_ROBUSTNESS_PATH,
        "5.3.1 temporal robustness",
        [
            "metric", "horizon",
            "monthly_direction_agreement_pct",
            "monthly_sign_flips",
            "monthly_gap_pct_min",
            "monthly_gap_pct_max",
        ],
    )

if "horizon_robustness_df" not in globals():
    horizon_robustness_df = load_saved_table(
        HORIZON_ROBUSTNESS_PATH,
        "5.3.1 horizon robustness",
        [
            "metric",
            "h1_gap_pct", "h2_gap_pct", "h5_gap_pct",
            "aggregate_horizons_same_direction",
            "aggregate_horizon_spread_pp",
            "row_direction_agreement_pct",
            "median_row_horizon_spread_pct",
        ],
    )

if "pre_cp_error_calibration_df" not in globals():
    pre_cp_error_calibration_df = load_saved_table(
        PRE_CP_CALIBRATION_PATH,
        "5.3.1 Pre-CP error calibration",
        [
            "metric", "horizon",
            "median_absolute_mae_units",
            "p90_absolute_mae_units",
            "share_abs_ge_1_mae_pct",
            "share_abs_ge_2_mae_pct",
        ],
    )

if "predecision_evidence_df" not in globals():
    predecision_evidence_df = load_saved_table(
        PREDECISION_EVIDENCE_PATH,
        "5.3.1 pre-Decision evidence",
        [
            "metric", "horizon",
            "primary_gap_pct",
            "population_same_direction",
            "population_gap_shift_pp",
            "speed_weight_same_direction",
            "speed_weight_gap_shift_pp",
            "monthly_direction_agreement_pct",
            "geography_min_direction_agreement_pct",
            "geography_mean_direction_agreement_pct",
            "aggregate_horizons_same_direction",
            "row_direction_agreement_pct",
            "aggregate_horizon_spread_pp",
            "median_absolute_mae_units",
            "p90_absolute_mae_units",
        ],
    )


# ---------------------------------------------------------------------
# Add detail needed by downstream robustness views
# ---------------------------------------------------------------------

population_detail_df = population_robustness_df[
    [
        "metric", "horizon",
        "native_gap_pct",
        "absolute_gap_pct_shift_pp",
    ]
]

speed_detail_df = speed_weighting_robustness_df[
    ["metric", "horizon", "common_observed_gap_pct"]
]

temporal_detail_df = temporal_robustness_df[
    [
        "metric", "horizon",
        "monthly_sign_flips",
        "monthly_gap_pct_min",
        "monthly_gap_pct_max",
    ]
]

horizon_detail_df = horizon_robustness_df[
    [
        "metric",
        "h1_gap_pct", "h2_gap_pct", "h5_gap_pct",
        "median_row_horizon_spread_pct",
    ]
]

calibration_detail_df = pre_cp_error_calibration_df[
    [
        "metric", "horizon",
        "share_abs_ge_1_mae_pct",
        "share_abs_ge_2_mae_pct",
    ]
]


# ---------------------------------------------------------------------
# Build one authoritative row per Target × Horizon
# ---------------------------------------------------------------------

robustness_matrix_df = (
    predecision_evidence_df[
        [
            "metric", "horizon",
            "primary_gap_pct",
            "population_same_direction",
            "population_gap_shift_pp",
            "speed_weight_same_direction",
            "speed_weight_gap_shift_pp",
            "monthly_direction_agreement_pct",
            "geography_min_direction_agreement_pct",
            "geography_mean_direction_agreement_pct",
            "aggregate_horizons_same_direction",
            "row_direction_agreement_pct",
            "aggregate_horizon_spread_pp",
            "median_absolute_mae_units",
            "p90_absolute_mae_units",
        ]
    ]
    .merge(
        population_detail_df,
        on=["metric", "horizon"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        speed_detail_df,
        on=["metric", "horizon"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        temporal_detail_df,
        on=["metric", "horizon"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        horizon_detail_df,
        on="metric",
        how="left",
        validate="many_to_one",
    )
    .merge(
        calibration_detail_df,
        on=["metric", "horizon"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        decision_registry_df[
            [
                "metric", "horizon",
                "evidence_status",
                "baseline_result_summary",
                "caution_note",
                "supporting_view",
                "showcase_include",
            ]
        ],
        on=["metric", "horizon"],
        how="left",
        validate="one_to_one",
    )
)

robustness_matrix_df["Target"] = robustness_matrix_df["metric"].map(
    TARGET_LABELS
)
robustness_matrix_df["baseline_abs_gap_pct"] = robustness_matrix_df[
    "primary_gap_pct"
].abs()

robustness_matrix_df["baseline_gap_direction"] = np.select(
    [
        robustness_matrix_df["primary_gap_pct"].gt(0),
        robustness_matrix_df["primary_gap_pct"].lt(0),
    ],
    [
        "no_cp_above_observed",
        "no_cp_below_observed",
    ],
    default="approximately_equal",
)

robustness_matrix_df = (
    robustness_matrix_df
    .sort_values(["metric", "horizon"])
    .reset_index(drop=True)
)

if len(robustness_matrix_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Final robustness matrix does not contain all 15 jobs."
    )

if robustness_matrix_df[["metric", "horizon"]].duplicated().any():
    raise RuntimeError(
        "Final robustness matrix contains duplicate jobs."
    )


# ---------------------------------------------------------------------
# Build the compact reader-facing conclusion registry
# ---------------------------------------------------------------------

conclusion_registry_df = robustness_matrix_df[
    [
        "metric", "Target", "horizon",
        "evidence_status",
        "primary_gap_pct",
        "baseline_gap_direction",
        "baseline_result_summary",
        "supporting_view",
        "caution_note",
        "showcase_include",
    ]
].copy()


# ---------------------------------------------------------------------
# Reader-facing previews
# ---------------------------------------------------------------------
# The preview keeps the strongest comparison signals visible beside the
# frozen interpretation while the full matrix remains available downstream.

robustness_preview_df = robustness_matrix_df[
    [
        "Target", "horizon",
        "primary_gap_pct",
        "evidence_status",
        "population_gap_shift_pp",
        "speed_weight_gap_shift_pp",
        "monthly_direction_agreement_pct",
        "geography_min_direction_agreement_pct",
        "aggregate_horizons_same_direction",
        "median_absolute_mae_units",
    ]
].rename(columns={
    "horizon": "Horizon",
    "primary_gap_pct": "Baseline gap %",
    "evidence_status": "Evidence status",
    "population_gap_shift_pp": "Population shift pp",
    "speed_weight_gap_shift_pp": "Weight shift pp",
    "monthly_direction_agreement_pct": "Monthly agreement %",
    "geography_min_direction_agreement_pct": "Lowest geography agreement %",
    "aggregate_horizons_same_direction": "Horizons agree overall",
    "median_absolute_mae_units": "Median |MAE units|",
})

display(robustness_preview_df.round(3))
display(conclusion_registry_df.round(3))

,Target,Horizon,Baseline gap %,Evidence status,Population shift pp,Weight shift pp,Monthly agreement %,Lowest geography agreement %,Horizons agree overall,Median |MAE units|
0,FHVHV average speed,1,6.200,stable,0.001,-0.365,100.000,100.000,True,1.109
1,FHVHV average speed,2,6.257,stable,0.001,-0.252,100.000,100.000,True,1.098
2,FHVHV average speed,5,6.866,stable,0.001,-0.385,100.000,100.000,True,1.086
3,FHVHV trips,1,-10.997,stable,0.004,NaN,100.000,80.000,True,0.786
4,FHVHV trips,2,-11.028,stable,0.004,NaN,100.000,80.000,True,0.784
5,FHVHV trips,5,-11.028,stable,0.004,NaN,100.000,80.000,True,0.785
6,Subway ridership,1,-14.231,stable,-0.030,NaN,100.000,100.000,True,0.931
7,Subway ridership,2,-14.417,stable,-0.030,NaN,100.000,100.000,True,0.883
8,Subway ridership,5,-14.443,stable,-0.030,NaN,100.000,100.000,True,0.883
9,Taxi average speed,1,1.308,sensitive,0.000,1.873,60.000,40.000,True,0.610


,metric,Target,horizon,evidence_status,primary_gap_pct,baseline_gap_direction,baseline_result_summary,supporting_view,caution_note,showcase_include
0,fhvhv_avg_trip_speed,FHVHV average speed,1,stable,6.200,no_cp_above_observed,h=1: the no-CP forecast was 6.2% above the observed fhvhv average speed level over the full post-CP period.,"Weighting, time, geography, and forecast-horizon checks","The positive result remains consistent across the checks, and alternative trip weighting changes its size only modes...",True
1,fhvhv_avg_trip_speed,FHVHV average speed,2,stable,6.257,no_cp_above_observed,h=2: the no-CP forecast was 6.3% above the observed fhvhv average speed level over the full post-CP period.,"Weighting, time, geography, and forecast-horizon checks","The positive result remains consistent across the checks, and alternative trip weighting changes its size only modes...",True
2,fhvhv_avg_trip_speed,FHVHV average speed,5,stable,6.866,no_cp_above_observed,h=5: the no-CP forecast was 6.9% above the observed fhvhv average speed level over the full post-CP period.,"Weighting, time, geography, and forecast-horizon checks","The positive result remains consistent across the checks, and alternative trip weighting changes its size only modes...",True
3,fhvhv_trip_count,FHVHV trips,1,stable,-10.997,no_cp_below_observed,h=1: the no-CP forecast was 11.0% below the observed fhvhv trips level over the full post-CP period.,"Population, time, geography, and forecast-horizon checks","The negative result remains consistent when we change the supported population and look across time, geography, and ...",True
4,fhvhv_trip_count,FHVHV trips,2,stable,-11.028,no_cp_below_observed,h=2: the no-CP forecast was 11.0% below the observed fhvhv trips level over the full post-CP period.,"Population, time, geography, and forecast-horizon checks","The negative result remains consistent when we change the supported population and look across time, geography, and ...",True
5,fhvhv_trip_count,FHVHV trips,5,stable,-11.028,no_cp_below_observed,h=5: the no-CP forecast was 11.0% below the observed fhvhv trips level over the full post-CP period.,"Population, time, geography, and forecast-horizon checks","The negative result remains consistent when we change the supported population and look across time, geography, and ...",True
6,subway_ridership,Subway ridership,1,stable,-14.231,no_cp_below_observed,h=1: the no-CP forecast was 14.2% below the observed subway ridership level over the full post-CP period.,"Time, geography, and forecast-horizon checks","The negative result is very consistent across months, broader geographies, and forecast horizons.",True
7,subway_ridership,Subway ridership,2,stable,-14.417,no_cp_below_observed,h=2: the no-CP forecast was 14.4% below the observed subway ridership level over the full post-CP period.,"Time, geography, and forecast-horizon checks","The negative result is very consistent across months, broader geographies, and forecast horizons.",True
8,subway_ridership,Subway ridership,5,stable,-14.443,no_cp_below_observed,h=5: the no-CP forecast was 14.4% below the observed subway ridership level over the full post-CP period.,"Time, geography, and forecast-horizon checks","The negative result is very consistent across months, broader geographies, and forecast horizons.",True
9,taxi_avg_trip_speed,Taxi average speed,1,sensitive,1.308,no_cp_above_observed,h=1: the no-CP forecast was 1.3% above the observed taxi average speed level over the full post-CP period.,"Time, geography, weighting, and forecast-horizon checks","The overall positive gap is small, only 60% of months match that direction, and only 40% of borough groups do. The s...",True


**Findings.** The final robustness matrix contains all **15 Target × Horizon results** and places the numerical evidence used at the Decision Point beside each frozen interpretation. The smaller conclusion registry keeps the reader-facing result statement, evidence status, caution, supporting evidence view, and Showcase flag. Together, the two tables provide both the detailed evidence and the compact interpretation needed downstream.

**5.3.1.6C — Synthesize the final robustness story.**

The final matrix now contains several different kinds of evidence about each Target × Horizon result. Two views make that evidence easier to read together.

The first shows how consistently each result points in the same direction through time, across broader geography, and across forecast horizons, while also summarizing the population and speed-weighting checks. The second places the full-period counterfactual gap beside its Pre-CP forecast-error scale, connecting the size of the observed-versus-no-CP difference with the forecasting system's ordinary error.

In [15]:
# ---------------------------------------------------------------------
# 5.3.1.6C — Synthesize the final robustness story
# ---------------------------------------------------------------------

synthesis_df = robustness_matrix_df.copy()

# Keep the visual order consistent with the five-target forecasting contract.
synthesis_df["metric_order"] = pd.Categorical(
    synthesis_df["metric"],
    categories=FORECAST_TARGETS,
    ordered=True,
)

synthesis_df = (
    synthesis_df
    .sort_values(["metric_order", "horizon"])
    .reset_index(drop=True)
)

synthesis_df["Job"] = (
    synthesis_df["Target"]
    + " · h="
    + synthesis_df["horizon"].astype(str)
    + " · "
    + synthesis_df["evidence_status"].str.title()
)


# ---------------------------------------------------------------------
# View 1 — directional consistency across the main robustness dimensions
# ---------------------------------------------------------------------

agreement_specs = [
    (
        "monthly_direction_agreement_pct",
        "Time",
        BRAND_COLORS["dark_teal"],
    ),
    (
        "geography_min_direction_agreement_pct",
        "Geography",
        BRAND_COLORS["terracotta"],
    ),
    (
        "row_direction_agreement_pct",
        "Forecast horizon",
        BRAND_COLORS["seafoam"],
    ),
]

agreement_fig = go.Figure()

agreement_customdata = synthesis_df[
    ["Target", "horizon", "evidence_status"]
].to_numpy()

for column, label, color in agreement_specs:
    agreement_fig.add_trace(
        go.Scatter(
            x=synthesis_df[column],
            y=synthesis_df["Job"],
            mode="markers",
            name=label,
            marker={
                "size": 10,
                "color": color,
                "line": {
                    "width": 0.8,
                    "color": BRAND_COLORS["dark_teal"],
                },
            },
            customdata=agreement_customdata,
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "h=%{customdata[1]}<br>"
                f"{label} agreement: %{{x:.1f}}%<br>"
                "Final status: %{customdata[2]}"
                "<extra></extra>"
            ),
        )
    )

population_agree = int(
    synthesis_df["population_same_direction"]
    .fillna(False)
    .astype(bool)
    .sum()
)

speed_weight_mask = synthesis_df["speed_weight_same_direction"].notna()
speed_weight_total = int(speed_weight_mask.sum())
speed_weight_agree = int(
    synthesis_df.loc[
        speed_weight_mask,
        "speed_weight_same_direction",
    ]
    .astype(bool)
    .sum()
)

apply_branding(agreement_fig)

agreement_fig.update_layout(
    title={
        "text": (
            "<b>How Consistent Is Each Result Across the Evidence?</b>"
            "<br><sup>"
            f"Population direction agrees for {population_agree}/{EXPECTED_JOB_COUNT} jobs"
            f" · Speed-weight direction agrees for "
            f"{speed_weight_agree}/{speed_weight_total} speed jobs"
            "</sup>"
        ),
        "x": 0.5,
    },
    width=940,
    height=720,
    margin={"l": 235, "r": 40, "t": 110, "b": 65},
    xaxis={
        "title": "Directional agreement (%)",
        "range": [0, 102],
        "ticksuffix": "%",
    },
    yaxis={
        "title": "",
        "autorange": "reversed",
    },
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.02,
        "xanchor": "center",
        "x": 0.5,
    },
)

if RUN_VISUALS:
    agreement_fig.show()


# ---------------------------------------------------------------------
# View 2 — gap magnitude versus ordinary Pre-CP forecast error
# ---------------------------------------------------------------------

status_styles = {
    "stable": {
        "label": "Stable",
        "color": BRAND_COLORS["dark_teal"],
        "symbol": "circle",
    },
    "mixed": {
        "label": "Mixed",
        "color": BRAND_COLORS["terracotta"],
        "symbol": "diamond",
    },
    "sensitive": {
        "label": "Sensitive",
        "color": BRAND_COLORS["pale_peach"],
        "symbol": "x",
    },
}

magnitude_fig = go.Figure()

for status in ["stable", "mixed", "sensitive"]:
    status_df = synthesis_df.loc[
        synthesis_df["evidence_status"].eq(status)
    ].copy()

    if status_df.empty:
        continue

    style = status_styles[status]

    magnitude_fig.add_trace(
        go.Scatter(
            x=status_df["primary_gap_pct"],
            y=status_df["median_absolute_mae_units"],
            mode="markers",
            name=style["label"],
            marker={
                "size": 13,
                "color": style["color"],
                "symbol": style["symbol"],
                "line": {
                    "width": 1,
                    "color": BRAND_COLORS["dark_teal"],
                },
            },
            customdata=status_df[
                [
                    "Target",
                    "horizon",
                    "supporting_view",
                    "caution_note",
                ]
            ].to_numpy(),
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "h=%{customdata[1]}<br>"
                "Full-period gap: %{x:+.2f}%<br>"
                "Median absolute gap: %{y:.2f} Pre-CP MAE units<br>"
                "<br><b>Strongest evidence</b><br>%{customdata[2]}"
                "<br><br><b>Caution</b><br>%{customdata[3]}"
                "<extra></extra>"
            ),
        )
    )

apply_branding(magnitude_fig)

magnitude_fig.add_vline(
    x=0,
    line_width=1,
    line_dash="dash",
    line_color=BRAND_COLORS["dark_teal"],
)

magnitude_fig.update_layout(
    title={
        "text": (
            "<b>How Large Is the Gap Relative to Ordinary Forecast Error?</b>"
            "<br><sup>"
            "Full-period no-CP − observed gap versus "
            "median absolute Pre-CP validation-MAE units"
            "</sup>"
        ),
        "x": 0.5,
    },
    width=940,
    height=575,
    margin={"l": 85, "r": 40, "t": 110, "b": 75},
    xaxis={
        "title": "Full-period no-CP − observed gap (%)",
        "ticksuffix": "%",
        "zeroline": False,
    },
    yaxis={
        "title": "Median absolute gap (Pre-CP MAE units)",
        "rangemode": "tozero",
    },
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.02,
        "xanchor": "center",
        "x": 0.5,
    },
)

if RUN_VISUALS:
    magnitude_fig.show()

**Findings.** Looking across all of the evidence together, most results tell a consistent story. Subway ridership and both FHVHV measures are especially steady across time, geography, and forecast horizon, while Taxi trips keep a stable overall negative result despite more local variation. **Taxi average speed stands out as the least stable measure**, particularly at h=1. The comparison with normal Pre-CP forecast error also shows why gap size alone is not enough: some large citywide percentage gaps can still be modest relative to what the forecasting system normally misses by at an individual place and time.

### Section Summary

The final evidence review leaves **12 stable results, two mixed results, and one sensitive result**. The stable group includes all Taxi-trip, FHVHV-trip, FHVHV-speed, and Subway-ridership results. The uncertainty is concentrated in Taxi average speed: h=1 is sensitive, while h=2 and h=5 are mixed because the direction is steadier than the magnitude. The final robustness table and conclusion registry preserve both the headline result and the caution that should travel with it.

## 5.3.1.7 Validate and Export the Final Counterfactual Handoff

The final step publishes the completed validation evidence to authoritative `5.3.1.final_tables`.

The handoff keeps the detailed robustness dimensions separate while also providing the combined 15-job matrix and conclusion registry, giving downstream notebooks both drill-down evidence and ready-to-present interpretations.

**5.3.1.7A — Run final QA and publish the counterfactual validation package.**

Before writing the final handoff, confirm that every Target × Horizon is represented once, all Decision Point fields are complete, and each compact robustness table has the expected grain. The export then writes the validated package atomically and records every published artifact in one manifest.


In [16]:
# ---------------------------------------------------------------------
# 5.3.1.7A — Validate and export the final counterfactual package
# ---------------------------------------------------------------------

require_upstream_handoff()

if (
    "robustness_matrix_df" not in globals()
    or "conclusion_registry_df" not in globals()
):
    raise RuntimeError(
        "Run 5.3.1.6B before publishing the final 5.3.1 handoff."
    )


# ---------------------------------------------------------------------
# Load detailed robustness tables when needed for restart
# ---------------------------------------------------------------------

if "population_robustness_df" not in globals():
    population_robustness_df = pd.read_parquet(POPULATION_ROBUSTNESS_PATH)

if "speed_weighting_robustness_df" not in globals():
    speed_weighting_robustness_df = pd.read_parquet(
        SPEED_WEIGHTING_ROBUSTNESS_PATH
    )

if "temporal_robustness_df" not in globals():
    temporal_robustness_df = pd.read_parquet(TEMPORAL_ROBUSTNESS_PATH)

if "geography_robustness_df" not in globals():
    geography_robustness_df = pd.read_parquet(GEOGRAPHY_ROBUSTNESS_PATH)

if "horizon_robustness_df" not in globals():
    horizon_robustness_df = pd.read_parquet(HORIZON_ROBUSTNESS_PATH)

if "pre_cp_error_calibration_df" not in globals():
    pre_cp_error_calibration_df = pd.read_parquet(PRE_CP_CALIBRATION_PATH)


# ---------------------------------------------------------------------
# Define the final analytical handoff
# ---------------------------------------------------------------------

final_artifact_specs = {
    "counterfactual_robustness_matrix": {
        "frame": robustness_matrix_df,
        "path": FINAL_DIR / "counterfactual_robustness_matrix.parquet",
        "role": "primary_robustness_matrix",
    },
    "counterfactual_conclusion_registry": {
        "frame": conclusion_registry_df,
        "path": FINAL_DIR / "counterfactual_conclusion_registry.parquet",
        "role": "reader_facing_conclusions",
    },
    "counterfactual_population_robustness": {
        "frame": population_robustness_df,
        "path": FINAL_DIR / "counterfactual_population_robustness.parquet",
        "role": "sensitivity_detail",
    },
    "counterfactual_speed_weighting_robustness": {
        "frame": speed_weighting_robustness_df,
        "path": FINAL_DIR / "counterfactual_speed_weighting_robustness.parquet",
        "role": "sensitivity_detail",
    },
    "counterfactual_temporal_robustness": {
        "frame": temporal_robustness_df,
        "path": FINAL_DIR / "counterfactual_temporal_robustness.parquet",
        "role": "sensitivity_detail",
    },
    "counterfactual_geography_robustness": {
        "frame": geography_robustness_df,
        "path": FINAL_DIR / "counterfactual_geography_robustness.parquet",
        "role": "sensitivity_detail",
    },
    "counterfactual_horizon_robustness": {
        "frame": horizon_robustness_df,
        "path": FINAL_DIR / "counterfactual_horizon_robustness.parquet",
        "role": "sensitivity_detail",
    },
    "counterfactual_pre_cp_calibration": {
        "frame": pre_cp_error_calibration_df,
        "path": FINAL_DIR / "counterfactual_pre_cp_calibration.parquet",
        "role": "pre_cp_error_context",
    },
}


# ---------------------------------------------------------------------
# Validate the final in-memory evidence
# ---------------------------------------------------------------------

qa_rows = []


def record_qa(check_id, passed, details):
    """Append one compact final-handoff QA result."""
    qa_rows.append({
        "check_id": check_id,
        "status": "PASS" if bool(passed) else "FAIL",
        "details": str(details),
    })


record_qa(
    "robustness_matrix_has_15_jobs",
    len(robustness_matrix_df) == EXPECTED_JOB_COUNT,
    f"rows={len(robustness_matrix_df)}; expected={EXPECTED_JOB_COUNT}",
)

record_qa(
    "robustness_matrix_unique_jobs",
    not robustness_matrix_df[["metric", "horizon"]].duplicated().any(),
    "metric × horizon must be unique",
)

record_qa(
    "conclusion_registry_has_15_jobs",
    len(conclusion_registry_df) == EXPECTED_JOB_COUNT,
    f"rows={len(conclusion_registry_df)}; expected={EXPECTED_JOB_COUNT}",
)

record_qa(
    "decision_status_complete",
    (
        conclusion_registry_df["evidence_status"].notna().all()
        & conclusion_registry_df["evidence_status"]
        .isin(ALLOWED_EVIDENCE_STATUS)
        .all()
    ),
    "every job must have stable, mixed, or sensitive status",
)

record_qa(
    "caution_notes_complete",
    conclusion_registry_df["caution_note"]
    .astype(str)
    .str.strip()
    .ne("")
    .all(),
    "every job must carry a reader-facing caution note",
)

record_qa(
    "supporting_views_complete",
    conclusion_registry_df["supporting_view"]
    .astype(str)
    .str.strip()
    .ne("")
    .all(),
    "every job must identify a strongest supporting view",
)

record_qa(
    "population_detail_complete",
    (
        len(population_robustness_df) == EXPECTED_JOB_COUNT
        and not population_robustness_df[["metric", "horizon"]]
        .duplicated()
        .any()
    ),
    f"rows={len(population_robustness_df)}",
)

record_qa(
    "speed_weighting_detail_complete",
    (
        len(speed_weighting_robustness_df)
        == len(SPEED_TARGETS) * len(FORECAST_HORIZONS)
    ),
    f"rows={len(speed_weighting_robustness_df)}",
)

record_qa(
    "temporal_detail_complete",
    (
        len(temporal_robustness_df) == EXPECTED_JOB_COUNT
        and not temporal_robustness_df[["metric", "horizon"]]
        .duplicated()
        .any()
    ),
    f"rows={len(temporal_robustness_df)}",
)

record_qa(
    "geography_detail_complete",
    len(geography_robustness_df) == EXPECTED_JOB_COUNT * 3,
    f"rows={len(geography_robustness_df)}",
)

record_qa(
    "horizon_detail_complete",
    (
        len(horizon_robustness_df) == len(FORECAST_TARGETS)
        and not horizon_robustness_df["metric"].duplicated().any()
    ),
    f"rows={len(horizon_robustness_df)}",
)

record_qa(
    "pre_cp_calibration_complete",
    (
        len(pre_cp_error_calibration_df) == EXPECTED_JOB_COUNT
        and not pre_cp_error_calibration_df[["metric", "horizon"]]
        .duplicated()
        .any()
    ),
    f"rows={len(pre_cp_error_calibration_df)}",
)

prewrite_qa_df = pd.DataFrame(qa_rows)

if prewrite_qa_df["status"].ne("PASS").any():
    display(prewrite_qa_df)
    raise RuntimeError(
        "Final 5.3.1 in-memory QA failed; publication is blocked."
    )


# ---------------------------------------------------------------------
# Publish the eight analytical artifacts
# ---------------------------------------------------------------------

FINAL_DIR.mkdir(parents=True, exist_ok=True)
manifest_rows = []

for artifact_id, spec in final_artifact_specs.items():
    frame = spec["frame"]
    path = spec["path"]

    atomic_write_parquet(frame, path)
    metadata = parquet_metadata(path)

    if metadata["rows"] != len(frame):
        raise RuntimeError(
            f"{artifact_id}: written row count does not match memory."
        )

    manifest_rows.append({
        "artifact_id": artifact_id,
        "role": spec["role"],
        "path": str(path),
        "rows": metadata["rows"],
        "columns": metadata["columns"],
        "contract_id": FINAL_HANDOFF_CONTRACT_ID,
    })


# ---------------------------------------------------------------------
# Validate the completed write and publish QA
# ---------------------------------------------------------------------

record_qa(
    "final_artifacts_written",
    all(spec["path"].exists() for spec in final_artifact_specs.values()),
    f"artifacts={len(final_artifact_specs)}",
)

final_qa_df = pd.DataFrame(qa_rows)

if final_qa_df["status"].ne("PASS").any():
    display(final_qa_df)
    raise RuntimeError(
        "Final 5.3.1 write validation failed; publication is blocked."
    )

atomic_write_parquet(final_qa_df, FINAL_QA_PATH)
qa_metadata = parquet_metadata(FINAL_QA_PATH)

manifest_rows.append({
    "artifact_id": "counterfactual_validation_qa",
    "role": "qa",
    "path": str(FINAL_QA_PATH),
    "rows": qa_metadata["rows"],
    "columns": qa_metadata["columns"],
    "contract_id": FINAL_HANDOFF_CONTRACT_ID,
})

final_handoff_manifest_df = (
    pd.DataFrame(manifest_rows)
    .sort_values("artifact_id")
    .reset_index(drop=True)
)

atomic_write_parquet(
    final_handoff_manifest_df,
    FINAL_MANIFEST_PATH,
)


# ---------------------------------------------------------------------
# Reader-facing confirmation
# ---------------------------------------------------------------------

display(final_qa_df)

display(
    final_handoff_manifest_df[
        [
            "artifact_id",
            "role",
            "rows",
            "columns",
            "contract_id",
        ]
    ]
)

,check_id,status,details
0,robustness_matrix_has_15_jobs,PASS,rows=15; expected=15
1,robustness_matrix_unique_jobs,PASS,metric × horizon must be unique
2,conclusion_registry_has_15_jobs,PASS,rows=15; expected=15
3,decision_status_complete,PASS,"every job must have stable, mixed, or sensitive status"
4,caution_notes_complete,PASS,every job must carry a reader-facing caution note
5,supporting_views_complete,PASS,every job must identify a strongest supporting view
6,population_detail_complete,PASS,rows=15
7,speed_weighting_detail_complete,PASS,rows=6
8,temporal_detail_complete,PASS,rows=15
9,geography_detail_complete,PASS,rows=45


,artifact_id,role,rows,columns,contract_id
0,counterfactual_conclusion_registry,reader_facing_conclusions,15,10,counterfactual_validation_handoff_v1
1,counterfactual_geography_robustness,sensitivity_detail,45,8,counterfactual_validation_handoff_v1
2,counterfactual_horizon_robustness,sensitivity_detail,5,9,counterfactual_validation_handoff_v1
3,counterfactual_population_robustness,sensitivity_detail,15,16,counterfactual_validation_handoff_v1
4,counterfactual_pre_cp_calibration,pre_cp_error_context,15,9,counterfactual_validation_handoff_v1
5,counterfactual_robustness_matrix,primary_robustness_matrix,15,35,counterfactual_validation_handoff_v1
6,counterfactual_speed_weighting_robustness,sensitivity_detail,6,14,counterfactual_validation_handoff_v1
7,counterfactual_temporal_robustness,sensitivity_detail,15,8,counterfactual_validation_handoff_v1
8,counterfactual_validation_qa,qa,13,3,counterfactual_validation_handoff_v1


**Findings.** All **13 final QA checks pass**. The final handoff publishes eight analytical tables—the combined robustness matrix, conclusion registry, and six detailed robustness or calibration tables—plus the QA table. The handoff manifest records those published artifacts under `counterfactual_validation_handoff_v1`, completing the package for downstream Showcase and reporting work.

## Section Summary

5.3.1 confirms that most of the observed-versus-no-CP results remain consistent when we change the supported population, look across time and geography, compare forecast horizons, and place the gaps against the model’s normal forecasting error. The final review freezes **12 stable, two mixed, and one sensitive** Target × Horizon results, with **Taxi average speed carrying the main uncertainty**. These labels describe how consistently the forecast-based counterfactual result holds up across the checks we tested; they should not be read as measures of causal certainty.

### 5.3.1.8 — Freeze a Self-Contained Counterfactual Showcase Package

The final Counterfactual Showcase package brings together both layers of Chapter 5 evidence:

* the observed-versus-no-CP gap surfaces and descriptive summaries from 5.2.1; and
* the robustness, calibration, and interpretation evidence from 5.3.1.

The authoritative 5.2.1 gap products are copied into `5.3.1.final_tables` without recalculation so the Showcase has one self-contained handoff. A portable manifest records exactly which files belong in that package.

In [17]:
# ---------------------------------------------------------------------
# 5.3.1.8 — Materialize the portable Counterfactual Showcase package
# ---------------------------------------------------------------------

if not WRITE_PORTABLE_SHOWCASE_PACKAGE:
    raise RuntimeError(
        "WRITE_PORTABLE_SHOWCASE_PACKAGE must be True to freeze the "
        "portable Counterfactual Showcase package."
    )


# ---------------------------------------------------------------------
# 1) Resolve the authoritative 5.2.1 files through its final manifest
# ---------------------------------------------------------------------

source_521_manifest_path = (
    FINAL_DIR.parent
    / "5.2.1.final_tables"
    / "counterfactual_gap_handoff_manifest.parquet"
)

if not source_521_manifest_path.exists():
    raise FileNotFoundError(
        "The authoritative 5.2.1 handoff manifest is missing: "
        f"{source_521_manifest_path}"
    )

source_521_manifest_df = pd.read_parquet(
    source_521_manifest_path
)

if "path" not in source_521_manifest_df.columns:
    raise KeyError(
        "The 5.2.1 handoff manifest does not contain a 'path' column."
    )

source_521_paths_by_filename = {}

for raw_path in source_521_manifest_df["path"]:
    source_path = Path(str(raw_path))
    source_name = source_path.name

    if source_name in source_521_paths_by_filename:
        raise RuntimeError(
            "The 5.2.1 manifest contains duplicate filenames: "
            f"{source_name}"
        )

    source_521_paths_by_filename[source_name] = source_path


promoted_521_filenames = {
    "counterfactual_gap_surface.parquet",
    "counterfactual_horizon_stability.parquet",
    "counterfactual_temporal_summary.parquet",
    "counterfactual_spatial_summary.parquet",
    "counterfactual_mobility_profile_summary.parquet",
    "counterfactual_h1_feature_reliance.parquet",
    "counterfactual_gap_qa.parquet",
}

missing_521_files = sorted(
    promoted_521_filenames
    - set(source_521_paths_by_filename)
)

if missing_521_files:
    raise RuntimeError(
        "The 5.2.1 handoff is missing required Showcase files: "
        f"{missing_521_files}"
    )


# ---------------------------------------------------------------------
# 2) Restart-safe physical copy helper
# ---------------------------------------------------------------------

def materialize_portable_copy(source_path, target_path):
    """
    Copy one authoritative upstream artifact without materializing rows.

    This is especially important for the 7M-row gap surface: packaging
    should copy the already-validated Parquet file, not load and rewrite
    millions of rows merely to move it into the final Showcase package.
    """
    source_path = Path(source_path)
    target_path = Path(target_path)

    if not source_path.exists():
        raise FileNotFoundError(
            f"Portable source does not exist: {source_path}"
        )

    source_size = source_path.stat().st_size

    if (
        target_path.exists()
        and target_path.stat().st_size == source_size
    ):
        return "reused"

    temp_path = target_path.with_name(
        f"{target_path.name}.copying"
    )

    if temp_path.exists():
        temp_path.unlink()

    shutil.copy2(source_path, temp_path)

    if temp_path.stat().st_size != source_size:
        temp_path.unlink(missing_ok=True)

        raise RuntimeError(
            "Portable copy failed byte-size validation: "
            f"{source_path.name}"
        )

    temp_path.replace(target_path)

    return "copied"


# ---------------------------------------------------------------------
# 3) Promote the 5.2.1 analytical handoff into 5.3.1.final_tables
# ---------------------------------------------------------------------

portable_records = []

for filename in sorted(promoted_521_filenames):
    source_path = source_521_paths_by_filename[filename]
    target_path = FINAL_DIR / filename

    copy_status = materialize_portable_copy(
        source_path,
        target_path,
    )

    portable_records.append({
        "artifact_id": target_path.stem,
        "source_notebook": "5.2.1",
        "showcase_scope": (
            "provenance"
            if filename == "counterfactual_gap_qa.parquet"
            else "counterfactual_gap"
        ),
        "source_path": str(source_path),
        "portable_path": str(target_path),
        "copy_status": copy_status,
        "size_mb": (
            target_path.stat().st_size / (1024 ** 2)
        ),
        "purpose": (
            "Authoritative 5.2.1 counterfactual gap "
            "artifact carried forward for Showcase use."
        ),
    })


# ---------------------------------------------------------------------
# 4) Add the native 5.3.1 robustness and interpretation package
# ---------------------------------------------------------------------

native_531_specs = {
    "counterfactual_robustness_matrix.parquet":
        "Complete 15-job robustness evidence surface.",
    "counterfactual_conclusion_registry.parquet":
        "Reader-facing Stable / Mixed / Sensitive registry.",
    "counterfactual_population_robustness.parquet":
        "Primary-versus-native population sensitivity.",
    "counterfactual_speed_weighting_robustness.parquet":
        "Average-speed weighting sensitivity.",
    "counterfactual_temporal_robustness.parquet":
        "Temporal consistency evidence.",
    "counterfactual_geography_robustness.parquet":
        "Geographic consistency evidence.",
    "counterfactual_horizon_robustness.parquet":
        "Cross-horizon consistency evidence.",
    "counterfactual_pre_cp_calibration.parquet":
        "Post-CP gaps calibrated to ordinary Pre-CP forecast error.",
    "counterfactual_validation_qa.parquet":
        "Final 5.3.1 validation QA.",
    "counterfactual_validation_handoff_manifest.parquet":
        "Original analytical 5.3.1 robustness handoff manifest.",
}

for filename, purpose in native_531_specs.items():
    artifact_path = FINAL_DIR / filename

    if not artifact_path.exists():
        raise FileNotFoundError(
            "Required 5.3.1 artifact is missing: "
            f"{artifact_path}"
        )

    portable_records.append({
        "artifact_id": artifact_path.stem,
        "source_notebook": "5.3.1",
        "showcase_scope": (
            "provenance"
            if filename in {
                "counterfactual_validation_qa.parquet",
                "counterfactual_validation_handoff_manifest.parquet",
            }
            else "counterfactual_robustness"
        ),
        "source_path": str(artifact_path),
        "portable_path": str(artifact_path),
        "copy_status": "native",
        "size_mb": (
            artifact_path.stat().st_size / (1024 ** 2)
        ),
        "purpose": purpose,
    })


# ---------------------------------------------------------------------
# 5) Freeze one physical transfer manifest for the Showcase
# ---------------------------------------------------------------------

counterfactual_portable_manifest_df = (
    pd.DataFrame(portable_records)
    .sort_values(
        ["showcase_scope", "source_notebook", "artifact_id"]
    )
    .reset_index(drop=True)
)

if counterfactual_portable_manifest_df[
    "artifact_id"
].duplicated().any():
    duplicate_ids = (
        counterfactual_portable_manifest_df.loc[
            counterfactual_portable_manifest_df[
                "artifact_id"
            ].duplicated(keep=False),
            "artifact_id",
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    raise RuntimeError(
        "Portable counterfactual manifest contains duplicate IDs: "
        f"{sorted(duplicate_ids)}"
    )

missing_portable_paths = [
    path
    for path in counterfactual_portable_manifest_df[
        "portable_path"
    ]
    if not Path(path).exists()
]

if missing_portable_paths:
    raise RuntimeError(
        "Portable counterfactual package contains missing files: "
        f"{missing_portable_paths}"
    )

manifest_temp_path = (
    COUNTERFACTUAL_SHOWCASE_PORTABLE_MANIFEST_PATH.with_name(
        f"{COUNTERFACTUAL_SHOWCASE_PORTABLE_MANIFEST_PATH.name}.tmp"
    )
)

counterfactual_portable_manifest_df.to_parquet(
    manifest_temp_path,
    index=False,
)
manifest_temp_path.replace(
    COUNTERFACTUAL_SHOWCASE_PORTABLE_MANIFEST_PATH
)

display(
    counterfactual_portable_manifest_df[
        [
            "artifact_id",
            "source_notebook",
            "showcase_scope",
            "copy_status",
            "size_mb",
            "portable_path",
        ]
    ]
)

print(
    "\nPortable counterfactual package frozen:"
    f"\n  artifacts listed = "
    f"{len(counterfactual_portable_manifest_df):,}"
    f"\n  5.2.1 gap artifacts = "
    f"{counterfactual_portable_manifest_df['source_notebook'].eq('5.2.1').sum():,}"
    f"\n  5.3.1 evidence artifacts = "
    f"{counterfactual_portable_manifest_df['source_notebook'].eq('5.3.1').sum():,}"
    f"\n  total size = "
    f"{counterfactual_portable_manifest_df['size_mb'].sum() / 1024:.3f} GB"
    f"\n  manifest = "
    f"{COUNTERFACTUAL_SHOWCASE_PORTABLE_MANIFEST_PATH}",
    flush=True,
)

,artifact_id,source_notebook,showcase_scope,copy_status,size_mb,portable_path
0,counterfactual_gap_surface,5.2.1,counterfactual_gap,reused,286.249,/datasets/_deepnote_work/pipeline_data/5.3.1.final_tables/counterfactual_gap_surface.parquet
1,counterfactual_h1_feature_reliance,5.2.1,counterfactual_gap,reused,0.007,/datasets/_deepnote_work/pipeline_data/5.3.1.final_tables/counterfactual_h1_feature_reliance.parquet
2,counterfactual_horizon_stability,5.2.1,counterfactual_gap,reused,184.563,/datasets/_deepnote_work/pipeline_data/5.3.1.final_tables/counterfactual_horizon_stability.parquet
3,counterfactual_mobility_profile_summary,5.2.1,counterfactual_gap,reused,0.043,/datasets/_deepnote_work/pipeline_data/5.3.1.final_tables/counterfactual_mobility_profile_summary.parquet
4,counterfactual_spatial_summary,5.2.1,counterfactual_gap,reused,0.114,/datasets/_deepnote_work/pipeline_data/5.3.1.final_tables/counterfactual_spatial_summary.parquet
5,counterfactual_temporal_summary,5.2.1,counterfactual_gap,reused,0.018,/datasets/_deepnote_work/pipeline_data/5.3.1.final_tables/counterfactual_temporal_summary.parquet
6,counterfactual_conclusion_registry,5.3.1,counterfactual_robustness,native,0.010,/datasets/_deepnote_work/pipeline_data/5.3.1.final_tables/counterfactual_conclusion_registry.parquet
7,counterfactual_geography_robustness,5.3.1,counterfactual_robustness,native,0.007,/datasets/_deepnote_work/pipeline_data/5.3.1.final_tables/counterfactual_geography_robustness.parquet
8,counterfactual_horizon_robustness,5.3.1,counterfactual_robustness,native,0.007,/datasets/_deepnote_work/pipeline_data/5.3.1.final_tables/counterfactual_horizon_robustness.parquet
9,counterfactual_population_robustness,5.3.1,counterfactual_robustness,native,0.013,/datasets/_deepnote_work/pipeline_data/5.3.1.final_tables/counterfactual_population_robustness.parquet



Portable counterfactual package frozen:
  artifacts listed = 17
  5.2.1 gap artifacts = 7
  5.3.1 evidence artifacts = 10
  total size = 0.460 GB
  manifest = /datasets/_deepnote_work/pipeline_data/5.3.1.final_tables/counterfactual_showcase_portable_manifest.parquet


**Findings.** The final Chapter 5 Showcase package now contains both sides of the counterfactual story: the observed-versus-no-CP results from 5.2.1 and the robustness and interpretation evidence from 5.3.1. It also carries the h=1 feature-reliance comparison that shows how the forecasting information mix changes inside the synthetic no-CP world.

`counterfactual_showcase_portable_manifest.parquet` is the authoritative checklist for the counterfactual files that should move into the Showcase.

In [18]:
from pathlib import Path
import zipfile

import pandas as pd


# ---------------------------------------------------------------------
# One-click Showcase export bundle
# ---------------------------------------------------------------------
#
# The portable manifests are the authoritative transfer checklists.
# This cell packages only those files, plus the manifests themselves,
# into one ZIP that can be downloaded from Deepnote in a single click.

PIPELINE_DATA_DIR = Path("pipeline_data")

FORECAST_DIR = PIPELINE_DATA_DIR / "4.7.1.final_tables"
COUNTERFACTUAL_DIR = PIPELINE_DATA_DIR / "5.3.1.final_tables"

FORECAST_MANIFEST = (
    FORECAST_DIR
    / "showcase_forecasting_portable_manifest.parquet"
)

COUNTERFACTUAL_MANIFEST = (
    COUNTERFACTUAL_DIR
    / "counterfactual_showcase_portable_manifest.parquet"
)

EXPORT_ZIP = (
    PIPELINE_DATA_DIR
    / "showcase_forecasting_counterfactual_export.zip"
)


def collect_portable_files(manifest_path):
    """
    Return every physical file listed by one portable manifest.

    Using the manifest keeps the export tied to the frozen handoff
    rather than accidentally zipping unrelated files from the folder.
    """
    manifest = pd.read_parquet(manifest_path)

    if "portable_path" not in manifest.columns:
        raise KeyError(
            f"{manifest_path.name} has no 'portable_path' column."
        )

    files = [
        Path(path)
        for path in manifest["portable_path"].astype(str)
    ]

    missing = [
        path
        for path in files
        if not path.exists()
    ]

    if missing:
        raise FileNotFoundError(
            "Portable manifest references missing files:\n"
            + "\n".join(str(path) for path in missing)
        )

    return files


forecast_files = collect_portable_files(
    FORECAST_MANIFEST
)

counterfactual_files = collect_portable_files(
    COUNTERFACTUAL_MANIFEST
)

# Include the transfer manifests themselves too.
export_groups = {
    "4.7.1.final_tables": (
        forecast_files
        + [FORECAST_MANIFEST]
    ),
    "5.3.1.final_tables": (
        counterfactual_files
        + [COUNTERFACTUAL_MANIFEST]
    ),
}


# ---------------------------------------------------------------------
# Build one ZIP.
#
# ZIP_STORED is intentional: Parquet already uses compression, so
# recompressing it usually costs substantial time for little benefit.
# ---------------------------------------------------------------------

if EXPORT_ZIP.exists():
    EXPORT_ZIP.unlink()

written_sources = set()

with zipfile.ZipFile(
    EXPORT_ZIP,
    mode="w",
    compression=zipfile.ZIP_STORED,
    allowZip64=True,
) as archive:

    for folder_name, files in export_groups.items():

        for source_path in files:
            source_path = Path(source_path).resolve()

            # Avoid adding the same physical file twice.
            if source_path in written_sources:
                continue

            written_sources.add(source_path)

            archive_name = (
                Path(folder_name)
                / source_path.name
            )

            archive.write(
                source_path,
                arcname=archive_name,
            )


zip_size_gb = (
    EXPORT_ZIP.stat().st_size
    / (1024 ** 3)
)

print(
    "Showcase export bundle ready."
    f"\n  Files packaged: {len(written_sources):,}"
    f"\n  ZIP size: {zip_size_gb:.3f} GB"
    f"\n  Download: {EXPORT_ZIP}"
)

Showcase export bundle ready.
  Files packaged: 35
  ZIP size: 0.665 GB
  Download: pipeline_data/showcase_forecasting_counterfactual_export.zip


## Notebook Close

The counterfactual validation workflow is complete. The final outputs preserve the detailed robustness evidence, the frozen Target × Horizon interpretations, and the cautions needed to present those results responsibly. Downstream notebooks and the Showcase should consume the files in `5.3.1.final_tables`.

## Final File Inventory

The native 5.3.1 analytical handoff contains the following robustness, interpretation, and validation files. The portable Showcase package additionally carries the seven authoritative 5.2.1 counterfactual artifacts; those combined files are tracked by `counterfactual_showcase_portable_manifest.parquet`.

| File | Purpose | Rows | Columns |
|---|---|---:|---:|
| `counterfactual_robustness_matrix.parquet` | Complete Target × Horizon robustness evidence and final interpretation | 15 | 35 |
| `counterfactual_conclusion_registry.parquet` | Reader-facing Stable / Mixed / Sensitive conclusions and cautions | 15 | 10 |
| `counterfactual_population_robustness.parquet` | Check whether using a broader set of supported series changes the results | 15 | 16 |
| `counterfactual_speed_weighting_robustness.parquet` | Check whether alternative speed weighting changes the results | 6 | 14 |
| `counterfactual_temporal_robustness.parquet` | Check whether results keep the same direction across months | 15 | 8 |
| `counterfactual_geography_robustness.parquet` | Check whether results appear across broader geographic groups | 45 | 8 |
| `counterfactual_horizon_robustness.parquet` | Compare the same result across h=1, h=2, and h=5 | 5 | 9 |
| `counterfactual_pre_cp_calibration.parquet` | Compare post-CP gaps with ordinary Pre-CP forecasting error | 15 | 9 |
| `counterfactual_validation_qa.parquet` | Final validation checks for the 5.3.1 handoff | 13 | 3 |
| `counterfactual_validation_handoff_manifest.parquet` | Registry of the native 5.3.1 analytical handoff | 9 | 6 |

`counterfactual_showcase_portable_manifest.parquet` is the authoritative inventory for the combined Chapter 5 Showcase package.

In [7]:
import pandas as pd


path = (
    "pipeline_data/"
    "5.3.1.final_tables/"
    "counterfactual_gap_surface.parquet"
)

columns = [
    "metric",
    "horizon",
    "pre_cp_validation_system_mae",
    "counterfactual_gap",
    "counterfactual_gap_mae_units",
]

gap = pd.read_parquet(
    path,
    columns=columns,
)

mae_grain = (
    gap.groupby(
        ["metric", "horizon"],
        observed=True,
    )
    .agg(
        rows=("pre_cp_validation_system_mae", "size"),
        non_null_mae=("pre_cp_validation_system_mae", "count"),
        unique_mae=("pre_cp_validation_system_mae", "nunique"),
        min_mae=("pre_cp_validation_system_mae", "min"),
        median_mae=("pre_cp_validation_system_mae", "median"),
        max_mae=("pre_cp_validation_system_mae", "max"),
    )
    .reset_index()
    .sort_values(["metric", "horizon"])
)

print("MAE GRAIN")
print(mae_grain.to_string(index=False))


# Verify that this is actually the denominator used for ×MAE.
check = gap.loc[
    gap["counterfactual_gap"].notna()
    & gap["pre_cp_validation_system_mae"].notna()
    & gap["pre_cp_validation_system_mae"].gt(0)
    & gap["counterfactual_gap_mae_units"].notna()
].copy()

check["reconstructed_mae_units"] = (
    check["counterfactual_gap"]
    / check["pre_cp_validation_system_mae"]
)

check["absolute_difference"] = (
    check["reconstructed_mae_units"]
    - check["counterfactual_gap_mae_units"]
).abs()

print("\nDENOMINATOR CHECK")
print(
    check["absolute_difference"]
    .describe()
    .to_string()
)

MAE GRAIN
              metric  horizon   rows  non_null_mae  unique_mae    min_mae  median_mae    max_mae
fhvhv_avg_trip_speed        1 579535        579535           1   1.051031    1.051031   1.051031
fhvhv_avg_trip_speed        2 579278        579278           1   1.100604    1.100604   1.100604
fhvhv_avg_trip_speed        5 578507        578507           1   1.075755    1.075755   1.075755
    fhvhv_trip_count        1 581790        581790           1  58.929723   58.929723  58.929723
    fhvhv_trip_count        2 581532        581532           1  59.918516   59.918516  59.918516
    fhvhv_trip_count        5 580758        580758           1  59.849687   59.849687  59.849687
    subway_ridership        1 351780        351780           1 515.196112  515.196112 515.196112
    subway_ridership        2 351624        351624           1 544.093740  544.093740 544.093740
    subway_ridership        5 351156        351156           1 544.498596  544.498596 544.498596
 taxi_avg_trip_speed

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>